# MFEGSN — Pipeline Colab (Marker + LangExtract)

Notebook optimisé pour une exécution **pas à pas** sur Google Colab.
Il sépare clairement les **blocs texte** (explications) et les **blocs code** (exécution).

**Ordre recommandé :** Étapes 1 → 10, avec tests optionnels si besoin.

## Étape 1 — Installer les dépendances
- Systèmes : zstd (requis pour Ollama).
- Python : marker-pdf, langextract, pillow.

In [ ]:
# ============================================================================
# VERSION FINALE CORRIGÉE
# ============================================================================
!apt-get update -qq && apt-get install -y ghostscript poppler-utils tesseract-ocr tesseract-ocr-fra tesseract-ocr-eng tesseract-ocr-ara unpaper pngquant -qq
!rm -rf /usr/lib/python3/dist-packages/pdfminer.six*
!python -m pip install -q --upgrade pip

# --- ÉTAPE 1 : Installer pypdfium2 et pydantic SEULS ---
!python -m pip install -q "pypdfium2==4.30.0" "pydantic<2.12.4"

# --- ÉTAPE 2 : Installer marker-pdf avec TOUTES ses dépendances ---
# (on laisse pip gérer les dépendances, sauf Pillow qu'on va forcer après)
!python -m pip install -q marker-pdf

# --- ÉTAPE 3 : Installer EasyOCR et ses dépendances (torch, torchvision, etc.) ---
# (ceci va installer Pillow 12.x, mais on va le corriger après)
!python -m pip install -q easyocr

# --- ÉTAPE 4 : OCRmyPDF ---
!python -m pip install -q "ocrmypdf<17.0.0"

# --- ÉTAPE 5 : Autres outils ---
!python -m pip install -q langextract google-generativeai PyPDF2 pypdf psutil

# --- ÉTAPE 6 : Plugin OCRmyPDF-EasyOCR ---
try:
    !python -m pip install -q ocrmypdf-easyocr
    print("✅ OCRmyPDF-EasyOCR installé")
except:
    print("⚠️ OCRmyPDF-EasyOCR non disponible")

# --- FIX CRITIQUE : FORCER PILLOW 10.4.0 EN TOUT DERNIER ---
# --no-deps empêche pip de réinstaller torch/torchvision qui ramèneraient Pillow 12.x
print("\n🔧 Correction finale de Pillow...")
!python -m pip install --force-reinstall --no-deps "pillow==10.4.0"

# --- VÉRIFICATION FINALE ---
print("\n🔍 Versions critiques installées :")
!python -c "import PIL; print(f'✅ Pillow: {PIL.__version__}')"
!python -c "import pypdfium2; print(f'✅ pypdfium2: {pypdfium2.__version__}')"
!python -c "import torch; print(f'✅ PyTorch: {torch.__version__}')"

print("\n✅ Installation terminée.")

## Étape 2 — Monter Google Drive
Exécutez cette cellule si vos PDF sont sur Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
from pathlib import Path

# === CONFIGURATION DU CACHE SUR DRIVE ===
# Cela permet de sauvegarder les modèles (HuggingFace, Marker, etc.) 
# sur Drive pour ne pas les retélécharger à chaque session.

# Dossier de cache
DRIVE_CACHE_BASE = "/content/drive/MyDrive/.mfegsn_cache" 
DRIVE_CACHE_DIR = Path(DRIVE_CACHE_BASE)
DRIVE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Sous-dossiers
HF_CACHE_DRIVE = DRIVE_CACHE_DIR / "huggingface"
TORCH_CACHE_DRIVE = DRIVE_CACHE_DIR / "torch"
DATALAB_CACHE_DRIVE = DRIVE_CACHE_DIR / "datalab"

for cache_dir in [HF_CACHE_DRIVE, TORCH_CACHE_DRIVE, DATALAB_CACHE_DRIVE]:
    cache_dir.mkdir(parents=True, exist_ok=True)

# Variables d'environnement pour rediriger le cache
os.environ["HF_HOME"] = str(HF_CACHE_DRIVE)
os.environ["TRANSFORMERS_CACHE"] = str(HF_CACHE_DRIVE)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_CACHE_DRIVE)
os.environ["TORCH_HOME"] = str(TORCH_CACHE_DRIVE)
os.environ["XDG_CACHE_HOME"] = str(DRIVE_CACHE_DIR)

# Information
def get_dir_size(path):
    total = 0
    if path.exists():
        for f in path.rglob("*"):
            if f.is_file():
                total += f.stat().st_size
    return total / 1e9

cache_size = get_dir_size(DRIVE_CACHE_DIR)
print("="*60)
print("✅ Drive monté et Cache configuré")
print(f"📂 Cache: {DRIVE_CACHE_DIR}")
print(f"💾 Taille: {cache_size:.2f} GB")
print("="*60)

## Étape 3 — Configurer les dossiers
Modifiez les chemins ci-dessous selon votre Drive.

In [ ]:
from pathlib import Path
import shutil
import os

# === MODIFIEZ CES CHEMINS (Vers vos dossiers Drive) ===
DRIVE_INPUT_DIR = Path("/content/drive/MyDrive/PHDM/ALL/ALLPDF")
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/PHDM/ALL/ALLMD")

# === CONFIGURATION LOCALE (Optimisation Colab) ===
# On travaille en local pour éviter les lenteurs de Drive
INPUT_DIR = Path("/content/work/ALLPDF")
OUTPUT_DIR = Path("/content/work/ALLMD")

print("=" * 60)
print("📂 PRÉPARATION DE L'ENVIRONNEMENT DE TRAVAIL")
print("=" * 60)

# 1. Création des dossiers locaux
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 1b. Fonction pour créer la structure de dossiers par fichier
def get_doc_dirs(doc_name):
    """
    Crée la structure de dossiers pour un document donné :
    ALLMD/<doc_name>/
      ├── _ANALYSES/
      ├── _FIGURES/
      ├── _LOGS/
      └── _REFERENCES/
    """
    doc_root = OUTPUT_DIR / doc_name
    dirs = {
        "root": doc_root,
        "analyses": doc_root / "_ANALYSES",
        "figures": doc_root / "_FIGURES",
        "logs": doc_root / "_LOGS",
        "references": doc_root / "_REFERENCES"
    }
    for p in dirs.values():
        p.mkdir(parents=True, exist_ok=True)
    return dirs

# NOTE: Les variables globales FIGURES_DIR, etc. sont supprimées 
# au profit de la structure par document générée par get_doc_dirs(doc_name).

# 2. Importation des PDFs depuis Drive
if DRIVE_INPUT_DIR.exists():
    print(f"📥 Copie des PDFs depuis : {DRIVE_INPUT_DIR}")
    pdfs = list(DRIVE_INPUT_DIR.glob("*.pdf"))
    print(f"   Nombre de fichiers identifiés : {len(pdfs)}")
    
    for i, pdf in enumerate(pdfs, 1):
        dest = INPUT_DIR / pdf.name
        # Copie si n'existe pas ou taille différente
        if not dest.exists() or dest.stat().st_size != pdf.stat().st_size:
            shutil.copy2(pdf, dest)
            if i % 10 == 0:
                print(f"   ... {i}/{len(pdfs)} copiés", end="\r")
    print(f"\n✅ Copie terminée. PDFs locaux : {len(list(INPUT_DIR.glob('*.pdf')))}")
else:
    print(f"⚠️ Dossier Drive introuvable : {DRIVE_INPUT_DIR}")

# Définition de la liste locale pour usage ultérieur
pdf_files = sorted(INPUT_DIR.glob("*.pdf"))
print(f"✅ Liste des fichiers PDF mise à jour : {len(pdf_files)} documents")

# 3. Préparation Dossiers Sortie Drive (Backup)
if not DRIVE_OUTPUT_DIR.exists():
    print(f"⚠️ Création du dossier sortie sur Drive : {DRIVE_OUTPUT_DIR}")
    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print(f"📂 Entrée locale : {INPUT_DIR}")
print(f"📂 Sortie locale : {OUTPUT_DIR}")
print(f"☁️  Backup Drive : {DRIVE_OUTPUT_DIR}")
print("=" * 60)

## Étape 4 — (Optionnel) Ollama + Gemma 3 4B
Activez uniquement si vous utilisez LangExtract avec un modèle local.

In [ ]:
USE_OLLAMA = False  # Mettre True si vous voulez Gemma via Ollama

if USE_OLLAMA:
    import subprocess
    import time

    # Installer Ollama (si besoin)
    !curl -fsSL https://ollama.com/install.sh | sh

    # Démarrer Ollama en arrière-plan
    ollama_process = subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )
    print("⏳ Démarrage d'Ollama...")
    time.sleep(10)

    # Télécharger Gemma 3 4B
    print("📥 Téléchargement de Gemma 3 4B (≈3GB)...")
    !ollama pull gemma3:4b
    !ollama list
    print("✅ Gemma 3 4B prêt !")

## Étape 5 — Configurer Marker
Réglages optimisés (3 workers parallèles pour 15GB VRAM + extraction figures + références).

In [ ]:
import os
import json
import re
import shutil
import base64
import psutil
import gc
from datetime import datetime
from pathlib import Path

import torch

# --- OPTIMISATION T4: CONFIGURATION ENV AVANT IMPORT MARKER ---
if torch.cuda.is_available():
    # Détection VRAM pour T4 (15GB) ou A100 (40GB)
    vram_available = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    # On laisse 2GB de marge pour le système/kernels
    marker_ram = max(4, int(vram_available - 2))
    os.environ["INFERENCE_RAM"] = str(marker_ram)
    print(f"⚡ Optimisation VRAM : INFERENCE_RAM={marker_ram}GB (Totale: {vram_available:.1f}GB)")

from PIL import Image
from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.config.parser import ConfigParser

# --- A. CONFIGURATION ADAPTATIVE ---
# ═══════════════════════════════════════════════════════════════════════════
# SCÉNARIO APPLIQUÉ : ADAPTATIF 300 DPI — QUALITÉ MOYENNE
# ═══════════════════════════════════════════════════════════════════════════
# - force_ocr ADAPTATIF → True pour scans/bad_ocr, False pour natifs
# - 300 DPI cible       → downscale automatique si >300 DPI (gain 20-30% temps)
# - Qualité moyenne     → use_llm désactivé, batch_size modéré
# - Objectif            → Éviter l'OCR inutile sur PDF natifs = gain VRAM + temps
# ═══════════════════════════════════════════════════════════════════════════

OCR_SCENARIO = "ADAPTIVE_300DPI_MEDIUM"

def get_system_resources():
    """Détecte les ressources système disponibles (RAM, VRAM)."""
    ram = psutil.virtual_memory().total / (1024**3)  # GB
    vram = 0
    device_name = "CPU"
    
    if torch.cuda.is_available():
        vram = torch.cuda.get_device_properties(0).total_memory / (1024**3) # GB
        device_name = torch.cuda.get_device_name(0)
    
    return {
        "ram_gb": ram,
        "vram_gb": vram,
        "device": device_name,
        "cuda": torch.cuda.is_available()
    }

resources = get_system_resources()
print(f"🖥️  Système détecté : {resources['device']}")
print(f"   RAM  : {resources['ram_gb']:.1f} GB")
print(f"   VRAM : {resources['vram_gb']:.1f} GB")

# ═══════════════════════════════════════════════════════════════════════════
# STRATÉGIE GPU-SAFE OPTIMISÉE POUR T4 vs A100
# ═══════════════════════════════════════════════════════════════════════════
# Paramètres initiaux (fallback)
workers = 2
batch_size = 4
strategy = "Standard"

if resources['vram_gb'] >= 35:
    # A100 (40GB)
    workers = 6
    batch_size = 12
    strategy = "A100 Beast Mode (6 workers, batch 12)"
elif resources['vram_gb'] >= 20: 
    # L4 / A10G (24GB)
    workers = 4
    batch_size = 8
    strategy = "High-End (4 workers, batch 8)"
elif resources['vram_gb'] >= 14:
    # T4 (15-16GB) - OPTIMUM
    # 2 workers = Stabilité (évite OOM sur gros PDF)
    # Batch 6 = Utilisation Tensor Cores
    workers = 2
    batch_size = 6
    strategy = "T4 Optimized (2 workers, batch 6)"
else:
    # CPU ou faible GPU
    workers = 1
    batch_size = 2
    strategy = "Low-VRAM / Safe Mode"

# --- Configuration de base partagée ---
_base_marker_config = {
    "workers": workers,
    "extract_images": True,
    "images_as_base64": False,
    "use_llm": False,
    "languages": ["fr", "en", "ar"],
    "paginate_output": True,
    "batch_size": batch_size,
}

# --- Config pour SCANS / BAD_OCR (force_ocr=True) ---
marker_config_ocr = {**_base_marker_config, "force_ocr": True}

# --- Config pour PDFs NATIFS (force_ocr=False) — gain T4 significatif ---
marker_config_native = {**_base_marker_config, "force_ocr": False}

# Garder marker_config comme alias pour rétro-compatibilité (défaut = OCR)
marker_config = marker_config_ocr

print(f"\n🎯 Scénario OCR : {OCR_SCENARIO}")
print(f"⚙️  Stratégie activée : {strategy}")
print(f"   Workers : {workers}")
print(f"   Batch   : {batch_size}")
print(f"   OCR     : ADAPTATIF (force_ocr=True pour scans, False pour natifs)")

if not resources['cuda']:
    print("⚠️ Attention : Exécution sur CPU détectée. Cela sera lent.")

print("📥 Chargement des modèles Marker...")

# Chargement des modèles UNE SEULE FOIS (partagés entre les deux converters)
model_dict = create_model_dict()

# --- Converter pour SCANS (force_ocr=True) ---
config_parser_ocr = ConfigParser(marker_config_ocr)
converter_ocr = PdfConverter(
    config=config_parser_ocr.generate_config_dict(),
    artifact_dict=model_dict,
)

# --- Converter pour NATIFS (force_ocr=False) → économie VRAM sur T4 ---
config_parser_native = ConfigParser(marker_config_native)
converter_native = PdfConverter(
    config=config_parser_native.generate_config_dict(),
    artifact_dict=model_dict,
)

# Alias par défaut (rétro-compatibilité)
converter = converter_ocr

print("✅ Marker configuré — 2 converters prêts (OCR + Natif).")

## Étape 6 — Fonctions utilitaires
Extraction des références, figures et conversion PDF → Markdown.

In [ ]:
import subprocess
import os
import time
from PIL import Image
import base64
import psutil
import gc
import json
import torch
import re
import string

# --- GESTION MÉMOIRE ---

def clear_memory():
    """Libère la mémoire RAM et VRAM inutilisée."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def get_memory_stats():
    """Retourne l'utilisation actuelle de la mémoire."""
    mem = psutil.virtual_memory()
    ram_used = mem.percent
    vram_used = 0
    if torch.cuda.is_available():
        vram_used = torch.cuda.memory_reserved(0) / (1024**3) # GB
    return f"RAM: {ram_used}% | VRAM: {vram_used:.1f}GB"

# --- DÉTECTION INTELLIGENTE DU TYPE DE PDF ---

def analyze_pdf_text_quality(pdf_path, sample_pages=5):
    """
    Analyse la qualité du texte extrait d'un PDF pour déterminer si l'OCR est nécessaire.
    
    Retourne un dict avec:
    - needs_ocr: bool - True si OCR recommandé
    - pdf_type: str - 'native', 'scan', 'mixed', 'empty'
    - confidence: float - 0.0 à 1.0
    - metrics: dict - détails des métriques analysées
    """
    from pypdf import PdfReader
    
    result = {
        "needs_ocr": True,  # Par défaut, on force l'OCR (safe)
        "pdf_type": "unknown",
        "confidence": 0.0,
        "metrics": {}
    }
    
    try:
        reader = PdfReader(str(pdf_path))
        total_pages = len(reader.pages)
        
        # Échantillonnage intelligent (début, milieu, fin)
        if total_pages <= sample_pages:
            pages_to_check = list(range(total_pages))
        else:
            pages_to_check = [
                0,  # Première page
                total_pages // 4,  # 25%
                total_pages // 2,  # Milieu
                3 * total_pages // 4,  # 75%
                total_pages - 1  # Dernière page
            ][:sample_pages]
        
        # Métriques agrégées
        chars_per_page = []
        alpha_ratios = []
        avg_word_lengths = []
        gibberish_scores = []
        
        for page_idx in pages_to_check:
            try:
                page = reader.pages[page_idx]
                text = page.extract_text() or ""
                
                # Nettoyage basique
                text = text.strip()
                
                # Métrique 1: Densité de caractères
                char_count = len(text)
                chars_per_page.append(char_count)
                
                if char_count < 10:
                    # Page quasi-vide, probablement une image
                    alpha_ratios.append(0.0)
                    avg_word_lengths.append(0.0)
                    gibberish_scores.append(1.0)
                    continue
                
                # Métrique 2: Ratio caractères alphanumériques
                alpha_chars = sum(1 for c in text if c.isalnum())
                alpha_ratio = alpha_chars / len(text) if len(text) > 0 else 0
                alpha_ratios.append(alpha_ratio)
                
                # Métrique 3: Longueur moyenne des mots
                words = text.split()
                if words:
                    avg_word_len = sum(len(w) for w in words) / len(words)
                    avg_word_lengths.append(avg_word_len)
                else:
                    avg_word_lengths.append(0.0)
                
                # Métrique 4: Score de "gibberish" (texte incohérent)
                gibberish_score = calculate_gibberish_score(text)
                gibberish_scores.append(gibberish_score)
                
            except Exception as e:
                # Page illisible, considérée comme scan
                chars_per_page.append(0)
                alpha_ratios.append(0.0)
                avg_word_lengths.append(0.0)
                gibberish_scores.append(1.0)
        
        # Calcul des moyennes
        avg_chars = sum(chars_per_page) / len(chars_per_page) if chars_per_page else 0
        avg_alpha_ratio = sum(alpha_ratios) / len(alpha_ratios) if alpha_ratios else 0
        avg_word_len = sum(avg_word_lengths) / len(avg_word_lengths) if avg_word_lengths else 0
        avg_gibberish = sum(gibberish_scores) / len(gibberish_scores) if gibberish_scores else 1.0
        
        # Stockage des métriques
        result["metrics"] = {
            "avg_chars_per_page": round(avg_chars, 1),
            "avg_alpha_ratio": round(avg_alpha_ratio, 3),
            "avg_word_length": round(avg_word_len, 2),
            "avg_gibberish_score": round(avg_gibberish, 3),
            "pages_analyzed": len(pages_to_check),
            "total_pages": total_pages
        }
        
        # Décision basée sur les heuristiques
        # ═══════════════════════════════════════════════════════════════
        
        # Cas 1: Très peu de texte → Scan (images)
        if avg_chars < 100:
            result["needs_ocr"] = True
            result["pdf_type"] = "scan"
            result["confidence"] = 0.95
            return result
        
        # Cas 2: Texte abondant et propre → PDF natif
        if avg_chars > 500 and avg_alpha_ratio > 0.7 and avg_gibberish < 0.2:
            result["needs_ocr"] = False
            result["pdf_type"] = "native"
            result["confidence"] = 0.9
            return result
        
        # Cas 3: Texte abondant mais bruité → OCR de mauvaise qualité antérieur
        if avg_chars > 300 and avg_gibberish > 0.4:
            result["needs_ocr"] = True
            result["pdf_type"] = "bad_ocr"
            result["confidence"] = 0.8
            return result
        
        # Cas 4: Texte moyen avec ratio alphanum correct → Probablement natif
        if avg_chars > 200 and avg_alpha_ratio > 0.6 and avg_word_len > 3 and avg_word_len < 15:
            result["needs_ocr"] = False
            result["pdf_type"] = "native"
            result["confidence"] = 0.75
            return result
        
        # Cas 5: Mixte ou incertain → Force OCR par sécurité
        result["needs_ocr"] = True
        result["pdf_type"] = "mixed"
        result["confidence"] = 0.6
        return result
        
    except Exception as e:
        # En cas d'erreur, on force l'OCR par sécurité
        result["needs_ocr"] = True
        result["pdf_type"] = "error"
        result["confidence"] = 0.5
        result["metrics"]["error"] = str(e)
        return result


def calculate_gibberish_score(text, min_words=10):
    """
    Calcule un score de "gibberish" (0 = texte propre, 1 = bruit total).
    
    Heuristiques:
    - Mots très courts ou très longs
    - Séquences de consonnes improbables
    - Caractères spéciaux excessifs
    - Répétitions anormales
    """
    if not text or len(text) < 20:
        return 1.0  # Texte trop court pour évaluer
    
    words = text.split()
    if len(words) < min_words:
        return 0.5  # Pas assez de mots pour une évaluation fiable
    
    penalties = 0.0
    checks = 0
    
    # Check 1: Mots de longueur anormale (< 2 ou > 20 caractères)
    abnormal_length = sum(1 for w in words if len(w) < 2 or len(w) > 20)
    penalty_length = abnormal_length / len(words)
    penalties += penalty_length
    checks += 1
    
    # Check 2: Ratio de caractères non-alphanumériques dans les mots
    word_text = "".join(words)
    non_alnum = sum(1 for c in word_text if not c.isalnum())
    penalty_special = non_alnum / len(word_text) if word_text else 0
    penalties += penalty_special * 2  # Poids plus élevé
    checks += 1
    
    # Check 3: Séquences de consonnes improbables (ex: "xzqwk")
    consonants = set("bcdfghjklmnpqrstvwxzBCDFGHJKLMNPQRSTVWXZ")
    long_consonant_seqs = 0
    for word in words:
        seq_len = 0
        for char in word:
            if char in consonants:
                seq_len += 1
                if seq_len >= 5:  # 5+ consonnes consécutives = suspect
                    long_consonant_seqs += 1
                    break
            else:
                seq_len = 0
    penalty_consonants = long_consonant_seqs / len(words) if words else 0
    penalties += penalty_consonants
    checks += 1
    
    # Check 4: Chiffres isolés ou séquences numériques excessives
    digit_words = sum(1 for w in words if w.isdigit())
    penalty_digits = digit_words / len(words) if len(words) > 0 else 0
    # Les chiffres isolés sont normaux jusqu'à un certain point
    penalties += max(0, penalty_digits - 0.1)
    checks += 1
    
    # Score final normalisé
    score = min(1.0, penalties / checks) if checks > 0 else 0.5
    return score


# --- ANALYSE PDF (mise à jour) ---

def get_pdf_complexity(pdf_path):
    """Calcule un score de complexité pour le tri."""
    try:
        size_mb = pdf_path.stat().st_size / (1024 * 1024)
        from pypdf import PdfReader
        try:
            with open(pdf_path, 'rb') as f:
                reader = PdfReader(f)
                num_pages = len(reader.pages)
        except:
            num_pages = size_mb * 2
            
        return size_mb * num_pages
    except:
        return 0

def check_pdf_health(pdf_path, max_size_mb=50, max_pages=100):
    """Vérifie si le PDF nécessite un traitement spécial (division) et analyse le besoin d'OCR."""
    warnings = []
    is_large = False
    ocr_analysis = None
    
    try:
        size_mb = pdf_path.stat().st_size / (1024 * 1024)
        
        from pypdf import PdfReader
        with open(pdf_path, 'rb') as f:
            reader = PdfReader(f)
            if reader.is_encrypted:
                warnings.append("Encrypted")
            num_pages = len(reader.pages)
            
        if size_mb > max_size_mb or num_pages > max_pages:
            is_large = True
            warnings.append(f"Large PDF ({size_mb:.1f}MB, {num_pages} pages)")
        
        # Analyse du besoin d'OCR
        ocr_analysis = analyze_pdf_text_quality(pdf_path)
        if ocr_analysis["pdf_type"] == "scan":
            warnings.append("Scan detected - OCR required")
        elif ocr_analysis["pdf_type"] == "bad_ocr":
            warnings.append("Poor OCR quality - re-OCR recommended")
            
    except Exception as e:
        warnings.append(f"Error reading: {str(e)}")
        
    return is_large, warnings, ocr_analysis

# --- SYNC / BACKUP (AMÉLIORÉ) ---

def verify_sync_for_document(doc_name, local_dir, drive_dir):
    """
    Vérifie qu'un document a bien été synchronisé vers Drive.
    Retourne (success: bool, details: dict)
    """
    details = {
        "doc_name": doc_name,
        "local_exists": False,
        "drive_exists": False,
        "files_synced": 0,
        "files_missing": [],
        "size_match": True
    }
    
    local_doc_dir = Path(local_dir) / doc_name
    drive_doc_dir = Path(drive_dir) / doc_name
    
    if not local_doc_dir.exists():
        details["error"] = "Local directory not found"
        return False, details
    
    details["local_exists"] = True
    details["drive_exists"] = drive_doc_dir.exists()
    
    if not drive_doc_dir.exists():
        details["error"] = "Drive directory not found"
        return False, details
    
    # Vérifier les fichiers
    local_files = list(local_doc_dir.rglob("*"))
    local_files = [f for f in local_files if f.is_file()]
    
    for local_file in local_files:
        relative_path = local_file.relative_to(local_doc_dir)
        drive_file = drive_doc_dir / relative_path
        
        if drive_file.exists():
            details["files_synced"] += 1
            # Vérifier la taille
            if local_file.stat().st_size != drive_file.stat().st_size:
                details["size_match"] = False
                details["files_missing"].append(f"{relative_path} (size mismatch)")
        else:
            details["files_missing"].append(str(relative_path))
    
    success = len(details["files_missing"]) == 0 and details["files_synced"] > 0
    return success, details


def sync_to_drive_with_verification(local_dir, drive_dir, doc_name=None, rsync=True):
    """
    Synchronise vers Drive avec vérification post-sync.
    
    Args:
        local_dir: Dossier source local
        drive_dir: Dossier destination Drive
        doc_name: Si spécifié, sync uniquement ce document
        rsync: Utiliser rsync si disponible
    
    Returns:
        (success: bool, message: str)
    """
    try:
        if doc_name:
            # Sync d'un document spécifique
            src = Path(local_dir) / doc_name
            dst = Path(drive_dir) / doc_name
            
            if not src.exists():
                return False, f"Source not found: {src}"
            
            dst.mkdir(parents=True, exist_ok=True)
            
            if rsync and shutil.which("rsync"):
                cmd = ["rsync", "-av", "--update", f"{src}/", f"{dst}/"]
                result = subprocess.run(cmd, capture_output=True, timeout=120)
                if result.returncode != 0:
                    return False, f"Rsync error: {result.stderr.decode()[:100]}"
            else:
                shutil.copytree(src, dst, dirs_exist_ok=True)
            
            # Vérification post-sync
            verified, details = verify_sync_for_document(doc_name, local_dir, drive_dir)
            if verified:
                return True, f"✅ {doc_name}: {details['files_synced']} fichiers synchro"
            else:
                return False, f"⚠️ {doc_name}: {len(details['files_missing'])} fichiers manquants"
        else:
            # Sync complet
            if rsync and shutil.which("rsync"):
                cmd = ["rsync", "-av", "--update", f"{local_dir}/", f"{drive_dir}/"]
                result = subprocess.run(cmd, capture_output=True, timeout=600)
                if result.returncode != 0:
                    return False, f"Rsync error: {result.stderr.decode()[:100]}"
            else:
                shutil.copytree(local_dir, drive_dir, dirs_exist_ok=True)
            
            return True, "✅ Sync complet terminé"
            
    except subprocess.TimeoutExpired:
        return False, "⚠️ Timeout sync"
    except Exception as e:
        return False, f"⚠️ Erreur sync: {str(e)[:100]}"


def periodic_sync_verification(local_dir, drive_dir, processed_docs, sample_size=5):
    """
    Vérification périodique d'un échantillon de documents synchro.
    
    Args:
        local_dir: Dossier local
        drive_dir: Dossier Drive
        processed_docs: Liste des documents traités
        sample_size: Nombre de documents à vérifier
    
    Returns:
        (all_ok: bool, report: dict)
    """
    import random
    
    report = {
        "checked": 0,
        "verified": 0,
        "failed": [],
        "timestamp": datetime.now().isoformat()
    }
    
    if not processed_docs:
        return True, report
    
    # Sélectionner un échantillon (documents récents + aléatoires)
    docs_to_check = []
    docs_list = list(processed_docs)
    
    # Toujours vérifier les 2 derniers
    if len(docs_list) >= 2:
        docs_to_check.extend(docs_list[-2:])
    elif docs_list:
        docs_to_check.append(docs_list[-1])
    
    # Ajouter des documents aléatoires
    remaining = [d for d in docs_list if d not in docs_to_check]
    if remaining and len(docs_to_check) < sample_size:
        sample_count = min(sample_size - len(docs_to_check), len(remaining))
        docs_to_check.extend(random.sample(remaining, sample_count))
    
    # Vérification
    for doc_name in docs_to_check:
        report["checked"] += 1
        success, details = verify_sync_for_document(doc_name, local_dir, drive_dir)
        
        if success:
            report["verified"] += 1
        else:
            report["failed"].append({
                "doc": doc_name,
                "reason": details.get("error", "Unknown"),
                "missing": details.get("files_missing", [])[:3]  # Limiter
            })
    
    all_ok = len(report["failed"]) == 0
    return all_ok, report


def resync_failed_documents(failed_docs, local_dir, drive_dir):
    """Tente de resynchroniser les documents échoués."""
    results = []
    
    for doc_info in failed_docs:
        doc_name = doc_info["doc"] if isinstance(doc_info, dict) else doc_info
        print(f"   🔄 Resync {doc_name}...")
        
        success, msg = sync_to_drive_with_verification(local_dir, drive_dir, doc_name)
        results.append({
            "doc": doc_name,
            "success": success,
            "message": msg
        })
        
        if success:
            print(f"      ✅ OK")
        else:
            print(f"      ❌ {msg}")
    
    return results


# Alias pour compatibilité avec l'ancien code
def sync_to_drive(local_dir, drive_dir, rsync=True):
    """Synchronise le dossier de sortie vers Google Drive (version simple)."""
    success, msg = sync_to_drive_with_verification(local_dir, drive_dir, rsync=rsync)
    if not success:
        print(f"⚠️ {msg}")
    return success

# --- FONCTIONS EXISTANTES AMÉLIORÉES ---

def extract_references_from_markdown(markdown_text):
    """Extrait la section références/bibliographie du Markdown."""
    references = {
        "references_text": "",
        "references_list": [],
        "reference_count": 0,
    }

    ref_patterns = [
        r"(?i)(?:^|\n)#{1,3}\s*(references|références|bibliography|bibliographie|works\s*cited|sources?)\s*[\s:]*\n([\s\S]*?)(?=\n#{1,3}\s|\Z)",
        r"(?i)(?:^|\n)\*\*(references|références|bibliography|bibliographie)\*\*\s*[\s:]*\n([\s\S]*?)(?=\n\*\*|\n#{1,3}|\Z)",
    ]

    for pattern in ref_patterns:
        match = re.search(pattern, markdown_text, re.MULTILINE)
        if match:
            ref_section = match.group(2).strip()
            references["references_text"] = ref_section

            ref_lines = []
            lines = ref_section.split("\n")
            current_ref = ""

            for line in lines:
                line = line.strip()
                if not line:
                    if current_ref:
                        ref_lines.append(current_ref.strip())
                        current_ref = ""
                    continue

                if re.match(r"^(\[\d+\]|\d+\.|[-•]|\([A-Za-z]\))", line):
                    if current_ref:
                        ref_lines.append(current_ref.strip())
                    current_ref = line
                else:
                    current_ref += " " + line

            if current_ref:
                ref_lines.append(current_ref.strip())

            references["references_list"] = [r for r in ref_lines if len(r) > 20]
            references["reference_count"] = len(references["references_list"])
            break

    return references


def extract_figures_info(markdown_text, images_dict):
    """Extrait les informations sur les figures du document."""
    figures = []
    fig_pattern = r"(?i)(figure|fig\.)\s*(\d+)?\s*[:]?\s*(.{0,120})"

    for match in re.finditer(fig_pattern, markdown_text):
        title = (match.group(3) or "").strip()
        figures.append({
            "label": match.group(0).strip(),
            "title": title,
        })

    if images_dict:
        for img_name in images_dict.keys():
            existing = any(f.get("path") == img_name for f in figures)
            if not existing:
                figures.append({
                    "label": str(img_name),
                    "title": "",
                    "path": str(img_name),
                })

    return figures


def save_figures(images_dict, doc_name, target_folder):
    """Sauvegarde les figures extraites dans le dossier spécifié."""
    if not images_dict:
        return []

    # Le dossier cible est déjà spécifique (ex: .../doc_name/_FIGURES)
    doc_figures_folder = target_folder
    doc_figures_folder.mkdir(parents=True, exist_ok=True)
    saved_paths = []

    for img_name, img_data in images_dict.items():
        safe_name = re.sub(r"[^a-zA-Z0-9_-]+", "_", str(img_name))
        img_path = doc_figures_folder / f"{safe_name}.png"

        try:
            if isinstance(img_data, Image.Image):
                img_data.save(img_path)
            elif isinstance(img_data, (bytes, bytearray)):
                with open(img_path, "wb") as f:
                    f.write(img_data)
            elif isinstance(img_data, str):
                if img_data.startswith("data:image"):
                    b64_data = img_data.split(",", 1)[1]
                    with open(img_path, "wb") as f:
                        f.write(base64.b64decode(b64_data))
                elif Path(img_data).exists():
                    shutil.copy(img_data, img_path)
                else:
                    with open(img_path, "wb") as f:
                        f.write(base64.b64decode(img_data))
            else:
                continue

            saved_paths.append(str(img_path))
        except Exception:
            continue

    return saved_paths

def markdown_to_ocr_text(markdown_path: Path, output_txt: Path):
    """Convertit le Markdown Marker en texte brut pour OCRmyPDF."""
    try:
        with open(markdown_path, 'r', encoding='utf-8') as f:
            content = f.read()
        
        # Nettoyage basique
        content = re.sub(r'!\[.*?\]\(.*?\)', '', content)
        content = re.sub(r'\[(.*?)\]\(.*?\)', r'\1', content)
        content = re.sub(r'```.*?```', '', content, flags=re.DOTALL)
        content = re.sub(r'#{1,6}\s+', '', content)
        
        with open(output_txt, 'w', encoding='utf-8') as f:
            f.write(content)
        return True
    except Exception as e:
        print(f"⚠️ Erreur conversion MD→TXT : {e}")
        return False

# ============================================================================
# RÉDUCTION DPI POUR SCANS HAUTE RÉSOLUTION (>300 DPI)
# ============================================================================
# Gain attendu : 20-30% réduction temps OCR pour scans 600 DPI.

def downscale_pdf_for_ocr(src_pdf: Path, max_width=2400) -> Path:
    """
    Réduit la résolution des pages d'un PDF si elles dépassent ~300 DPI (A4).
    
    Convertit chaque page en image, redimensionne si nécessaire,
    puis reconstruit un PDF temporaire optimisé pour OCR.
    
    Args:
        src_pdf: Chemin du PDF source
        max_width: Largeur max en pixels (~300 DPI pour A4 = 2400px)
    
    Returns:
        Path vers le PDF redimensionné (ou src_pdf si aucun redimensionnement nécessaire)
    """
    try:
        import pypdfium2 as pdfium
    except ImportError:
        return src_pdf  # Fallback: pas de redimensionnement
    
    try:
        pdf_doc = pdfium.PdfDocument(str(src_pdf))
        needs_downscale = False
        
        # Vérifier si au moins une page dépasse le seuil
        for i in range(min(3, len(pdf_doc))):  # Échantillonner 3 pages
            page = pdf_doc[i]
            # Rendu à 1x pour obtenir la taille native
            bitmap = page.render(scale=1)
            pil_img = bitmap.to_pil()
            if pil_img.width > max_width:
                needs_downscale = True
                break
        
        if not needs_downscale:
            pdf_doc.close()
            return src_pdf  # Pas besoin de redimensionner
        
        print(f"   📐 Réduction DPI détectée (>{max_width}px) → redimensionnement...")
        
        # Convertir toutes les pages en images redimensionnées
        downscaled_images = []
        for i in range(len(pdf_doc)):
            page = pdf_doc[i]
            bitmap = page.render(scale=1)
            pil_img = bitmap.to_pil()
            
            if pil_img.width > max_width:
                ratio = max_width / pil_img.width
                new_height = int(pil_img.height * ratio)
                pil_img = pil_img.resize((max_width, new_height), Image.LANCZOS)
            
            # Convertir en RGB si nécessaire (pour PDF)
            if pil_img.mode != 'RGB':
                pil_img = pil_img.convert('RGB')
            
            downscaled_images.append(pil_img)
        
        pdf_doc.close()
        
        # Sauvegarder comme nouveau PDF
        out_path = src_pdf.parent / f"{src_pdf.stem}_downscaled.pdf"
        if downscaled_images:
            downscaled_images[0].save(
                str(out_path), "PDF", save_all=True,
                append_images=downscaled_images[1:],
                resolution=300
            )
            print(f"   ✅ PDF redimensionné : {out_path.name} ({len(downscaled_images)} pages)")
            return out_path
        
        return src_pdf
        
    except Exception as e:
        print(f"   ⚠️ Erreur downscale DPI : {e} → utilisation du PDF original")
        return src_pdf

# Configuration globale pour OCR
USE_EASYOCR = True  # Mettre False pour forcer Tesseract

def check_easyocr_available():
    """Vérifie si le plugin OCRmyPDF-EasyOCR est disponible."""
    try:
        result = subprocess.run(
            ["ocrmypdf", "--plugin", "ocrmypdf_easyocr", "--help"],
            capture_output=True, timeout=10
        )
        return result.returncode == 0
    except:
        return False

# Cache pour éviter de vérifier à chaque appel
_EASYOCR_AVAILABLE = None

def make_searchable_pdf(src_pdf: Path, out_pdf: Path, sidecar_txt: Path = None, use_easyocr: bool = None):
    """
    Génère un PDF consultable (Searchable PDF) via OCRmyPDF.
    Applique automatiquement une réduction DPI si les scans sont >300 DPI.
    """
    global _EASYOCR_AVAILABLE
    
    # Réduction DPI pour scans haute résolution (gain 20-30% temps OCR)
    actual_src = downscale_pdf_for_ocr(Path(src_pdf))
    
    # Auto-détection EasyOCR si pas encore fait
    if _EASYOCR_AVAILABLE is None:
        _EASYOCR_AVAILABLE = check_easyocr_available()
        if _EASYOCR_AVAILABLE:
            print("   🔧 OCRmyPDF-EasyOCR disponible")
    
    # Décider du moteur
    if use_easyocr is None:
        use_easyocr = USE_EASYOCR and _EASYOCR_AVAILABLE
    
    max_retries = 2
    
    for attempt in range(max_retries + 1):
        try:
            # Construction de la commande de base
            cmd = [
                "ocrmypdf",
                "--optimize", "1",
                "--jobs", "2",
                "--output-type", "pdf",
                "--skip-big", "10",  # Skip pages avec images > 10 megapixels
            ]
            
            # Configuration du moteur OCR
            if use_easyocr and _EASYOCR_AVAILABLE:
                # EasyOCR via plugin
                cmd.extend([
                    "--plugin", "ocrmypdf_easyocr",
                    "-l", "fr+en+ar",  # Langues EasyOCR
                ])
            else:
                # Tesseract standard
                cmd.extend([
                    "--language", "fra+eng+ara",
                    "--tesseract-timeout", "300",
                ])
            
            # Options supplémentaires
            if sidecar_txt and sidecar_txt.exists():
                # Utiliser le texte Marker comme sidecar (améliore la précision)
                cmd.extend(["--sidecar", str(sidecar_txt)])
            
            # Option pour gérer les PDFs déjà avec texte
            cmd.append("--skip-text")
            
            # Fichiers source et destination (utilise actual_src si downscalé)
            cmd.extend([str(actual_src), str(out_pdf)])
            
            # Exécution avec timeout
            result = subprocess.run(
                cmd, 
                capture_output=True, 
                timeout=600,  # 10 minutes max par PDF
                text=True
            )
            
            if result.returncode == 0:
                return True, None
            elif result.returncode == 6:
                # Code 6 = le PDF contient déjà du texte, ce n'est pas une erreur
                return True, None
            else:
                # Erreur OCR
                err_msg = result.stderr or result.stdout
                
                # Si EasyOCR échoue, fallback sur Tesseract
                if use_easyocr and "easyocr" in err_msg.lower():
                    print(f"   ⚠️ EasyOCR échoué, fallback Tesseract...")
                    return make_searchable_pdf(src_pdf, out_pdf, sidecar_txt, use_easyocr=False)
                
                return False, err_msg[:200]  # Limiter la taille du message
            
        except subprocess.TimeoutExpired:
            print(f"   ⚠️ Timeout OCR (essai {attempt+1}/{max_retries+1})")
            clear_memory()
            continue
            
        except FileNotFoundError:
            return False, "OCRmyPDF non installé"
            
        except Exception as e:
            err_str = str(e).lower()
            if "memory" in err_str or "oom" in err_str:
                print(f"   ⚠️ OOM OCR (essai {attempt+1}/{max_retries+1})")
                clear_memory()
                continue
            return False, str(e)
    
    return False, "Échec après plusieurs tentatives"

def convert_pdf_complete(pdf_path, doc_name, force_ocr_override=None):
    """
    Conversion complète d'un PDF via Marker (nouvelle API PdfConverter).
    
    Scénario appliqué : ADAPTATIF 300 DPI QUALITÉ MOYENNE
    → force_ocr adaptatif selon le type de PDF détecté.
    → PDFs natifs (texte propre) : force_ocr=False (économie VRAM T4)
    → Scans / bad_ocr / mixtes : force_ocr=True (Surya OCR)
    
    Args:
        pdf_path: Chemin vers le PDF
        doc_name: Nom du document
        force_ocr_override: Si spécifié (True/False), force ce mode.
                            Si None, détection adaptative automatique.
    """
    result_data = {
        "doc_name": doc_name,
        "markdown_path": "",
        "figures": [],
        "figures_paths": [],
        "references": {},
        "searchable_pdf": "",
        "error": None,
        "ocr_used": True,
        "pdf_type": "unknown"
    }

    try:
        # Création de la structure de dossiers pour ce document
        doc_dirs = get_doc_dirs(doc_name)

        # ═══════════════════════════════════════════════════════════════
        # SCÉNARIO ADAPTATIF 300 DPI — ÉCONOMIE VRAM T4
        # ═══════════════════════════════════════════════════════════════
        # PDFs natifs → converter_native (force_ocr=False) = économie VRAM
        # Scans/bad_ocr → converter_ocr (force_ocr=True) = Surya OCR
        # ═══════════════════════════════════════════════════════════════
        if force_ocr_override is not None:
            use_force_ocr = force_ocr_override
        else:
            # Détection adaptative
            ocr_analysis = analyze_pdf_text_quality(pdf_path)
            use_force_ocr = ocr_analysis.get("needs_ocr", True)
            result_data["pdf_type"] = ocr_analysis.get("pdf_type", "unknown")
        
        if use_force_ocr:
            print(f"   🎯 Scénario: {OCR_SCENARIO} → force_ocr=True (scan/bad_ocr)")
            active_converter = converter_ocr
            result_data["ocr_used"] = True
        else:
            print(f"   🎯 Scénario: {OCR_SCENARIO} → force_ocr=False (PDF natif)")
            active_converter = converter_native
            result_data["ocr_used"] = False
            result_data["pdf_type"] = "native"

        # Utilisation du converter approprié (modèles partagés = pas de surcoût)
        rendered = active_converter(str(pdf_path))
        
        # Extraction du texte et des images
        full_text = rendered.markdown
        images = {}
        
        # Récupération des images depuis le rendu
        if hasattr(rendered, 'images') and rendered.images:
            images = rendered.images
        elif hasattr(rendered, 'children'):
            # Parcours des pages pour extraire les images
            for page_idx, page in enumerate(rendered.children):
                if hasattr(page, 'images'):
                    for img_idx, img in enumerate(page.images):
                        img_key = f"page_{page_idx}_img_{img_idx}"
                        if hasattr(img, 'image'):
                            images[img_key] = img.image
        
        # Sauvegarde du Markdown dans le dossier racine du document
        md_file = doc_dirs['root'] / f"{doc_name}.md"
        with open(md_file, "w", encoding='utf-8') as f:
            f.write(full_text)
            
        result_data['markdown_path'] = str(md_file)
        result_data['figures'] = extract_figures_info(full_text, images)
        result_data['figures_paths'] = save_figures(images, doc_name, doc_dirs['figures'])
        result_data['references'] = extract_references_from_markdown(full_text)
        
    except Exception as e:
        result_data['error'] = str(e)
        
    return result_data

## Étape 7 — LangExtract (optionnel)
Activez uniquement si vous souhaitez l'extraction structurée.

In [ ]:
USE_LANGEXTRACT = False  # Mettre True pour activer LangExtract

PROMPT_TEMPLATE = """
Vous êtes un assistant d'analyse pour des documents en sciences sociales.
Retournez un JSON structuré avec les sections suivantes :

1. CONTEXTE
- Thème principal
- Zone géographique
- Période

2. ACTEURS
- Institutions
- Pays
- Organisations

3. CONCEPTS CLÉS
- Mots-clés
- Concepts

4. DONNÉES
- Chiffres clés (si disponibles)

5. RÉFÉRENCES
- Principales références citées

6. FIGURES ET TABLEAUX
- Liste des figures mentionnées

Répondez uniquement avec un JSON valide.
"""

def _safe_json(obj):
    try:
        return json.loads(json.dumps(obj))
    except Exception:
        return {"raw": str(obj)}


def extract_with_langextract(markdown_text, doc_name, references_data=None, figures_data=None):
    """Extraction structurée avec LangExtract (optionnel)."""
    if not USE_LANGEXTRACT:
        return {"status": "skipped", "reason": "USE_LANGEXTRACT=False"}

    enriched_text = markdown_text

    if references_data and references_data.get("reference_count", 0) > 0:
        enriched_text += "\n\n## RÉFÉRENCES\n"
        enriched_text += f"Nombre de références : {references_data['reference_count']}\n"
        for i, ref in enumerate(references_data.get("references_list", [])[:20], 1):
            enriched_text += f"[{i}] {ref}\n"

    if figures_data:
        enriched_text += "\n\n## FIGURES IDENTIFIÉES\n"
        enriched_text += f"Nombre de figures : {len(figures_data)}\n"
        for fig in figures_data[:10]:
            enriched_text += f"- {fig.get('label', '')} {fig.get('title', '')}\n"

    try:
        import langextract as lx
        if hasattr(lx, "extract"):
            extraction = lx.extract(enriched_text, prompt=PROMPT_TEMPLATE)
        elif hasattr(lx, "LangExtract"):
            extractor = lx.LangExtract()
            extraction = extractor.extract(enriched_text, prompt=PROMPT_TEMPLATE)
        else:
            return {"status": "error", "error": "API LangExtract introuvable"}

        return _safe_json(extraction)
    except Exception as e:
        return {"status": "error", "error": str(e)}

## Étape 8a — Gestion intelligente des gros PDFs

### Division et fusion automatiques pour optimiser la mémoire

Cette section implémente une logique de **"Divide & Conquer"** pour traiter les documents volumineux qui causeraient des erreurs de mémoire (OOM) s'ils étaient traités en une seule fois.

**Logique du pipeline :**
```
📄 PDF Original
    ↓
1. 📝 MARKER → Markdown + Figures + Références
    ↓
2. 🔤 Conversion MD → TXT (nettoyage)
    ↓
3. 🔍 OCRmyPDF + Texte Marker → PDF Searchable
    ↓
4. 💾 Sauvegarde complète
```

**Critères de détection :**
- PDF > **50 pages** OU > **50 MB**

**Stratégie :**
1. **Division** : Découpage en chunks de **50 pages** (limite RAM ~12GB)
2. **Traitement** : Chaque chunk est converti indépendamment avec nettoyage VRAM
3. **Fusion** : Reconstitution via concaténation MD + PdfMerger
4. **Nettoyage** : Suppression des fichiers temporaires si succès

⚠️ **Important** : Exécutez cette cellule AVANT le test (Étape 8b).

In [ ]:
import sys
from datetime import datetime

# ============================================================================
# GUARD : VÉRIFICATION DES DÉPENDANCES (Cellules précédentes)
# ============================================================================
try:
    model_dict
    marker_config
except NameError:
    raise RuntimeError(
        "⚠️ ERREUR : Exécutez d'abord l'Étape 5 (Configuration Marker) !\n"
        "Les variables 'model_dict' et 'marker_config' sont requises."
    )

try:
    OUTPUT_DIR
    DRIVE_OUTPUT_DIR
except NameError:
    raise RuntimeError(
        "⚠️ ERREUR : Exécutez d'abord l'Étape 3 (Configuration dossiers) !\n"
        "Les variables 'OUTPUT_DIR' et 'DRIVE_OUTPUT_DIR' sont requises."
    )

try:
    convert_pdf_complete
    make_searchable_pdf
    clear_memory
    get_doc_dirs
    markdown_to_ocr_text
except NameError:
    raise RuntimeError(
        "⚠️ ERREUR : Exécutez d'abord l'Étape 6 (Fonctions utilitaires) !\n"
        "Les fonctions de conversion sont requises."
    )

print("✅ Dépendances vérifiées (Étapes 3, 5, 6)")

# ============================================================================
# GESTION ROBUSTE DES IMPORTS PYPDF / PYPDF2
# ============================================================================
try:
    from PyPDF2 import PdfReader, PdfWriter, PdfMerger
    print("📚 Utilisation de PyPDF2 (PdfMerger)")
except ImportError:
    try:
        from PyPDF2 import PdfFileReader as PdfReader, PdfFileWriter as PdfWriter, PdfFileMerger as PdfMerger
        print("📚 Utilisation de PyPDF2 (PdfFileMerger - ancienne API)")
    except ImportError:
        from pypdf import PdfReader, PdfWriter
        try:
            from pypdf import PdfMerger
        except ImportError:
            from pypdf import PdfFileMerger as PdfMerger
        print("📚 Utilisation de pypdf")

# ============================================================================
# CONSTANTES DE CONFIGURATION (Limites mémoire — OPTIMISÉES T4)
# ============================================================================
MAX_PAGES_IN_RAM = 50              # Limite stricte pour éviter OOM sur 12GB RAM
CHUNK_SIZE_PAGES = 50              # Taille de chunk unifiée (tous types de PDFs)
CHUNK_SIZE_PAGES_NATIVE = 50       # Taille de chunk pour PDFs natifs (confiance 90%+)
MAX_PDF_SIZE_MB = 50               # Seuil MB pour activer le mode chunking
MAX_PDF_PAGES = 50                 # Seuil pages pour activer le mode chunking
NATIVE_CONFIDENCE_THRESHOLD = 0.9  # Seuil de confiance pour déclencher force_ocr=False

print(f"⚙️  Configuration mémoire (T4 optimisé) :")
print(f"   - Max pages en RAM     : {MAX_PAGES_IN_RAM}")
print(f"   - Chunk unifié         : {CHUNK_SIZE_PAGES} pages")
print(f"   - Seuil chunking       : {MAX_PDF_SIZE_MB}MB ou {MAX_PDF_PAGES} pages")

# ============================================================================
# FONCTIONS DE DIVISION PDF
# ============================================================================

def get_pdf_page_count(pdf_path):
    """Récupère le nombre de pages d'un PDF de manière sûre."""
    try:
        reader = PdfReader(str(pdf_path))
        try:
            return len(reader.pages)
        except AttributeError:
            return reader.getNumPages()
    except Exception as e:
        print(f"⚠️ Erreur lecture pages : {e}")
        return 0

def split_pdf(pdf_path, chunk_size=CHUNK_SIZE_PAGES):
    """Divise un PDF en chunks de taille donnée. Retourne aussi les offsets de page."""
    chunks = []
    page_offsets = []
    temp_dir = pdf_path.parent / "temp_chunks" / pdf_path.stem
    temp_dir.mkdir(parents=True, exist_ok=True)
    
    try:
        reader = PdfReader(str(pdf_path))
        try:
            total_pages = len(reader.pages)
        except AttributeError:
            total_pages = reader.getNumPages()
        
        print(f"   📄 Document : {total_pages} pages → {(total_pages + chunk_size - 1) // chunk_size} chunks de {chunk_size} pages")
        
        for i in range(0, total_pages, chunk_size):
            writer = PdfWriter()
            end = min(i + chunk_size, total_pages)
            
            page_offsets.append(i)
            
            for page_num in range(i, end):
                try:
                    page = reader.pages[page_num]
                except AttributeError:
                    page = reader.getPage(page_num)
                writer.add_page(page)
            
            chunk_name = f"{pdf_path.stem}_chunk_{i//chunk_size + 1:03d}.pdf"
            chunk_path = temp_dir / chunk_name
            
            with open(chunk_path, "wb") as f:
                writer.write(f)
            chunks.append(chunk_path)
            
            del writer
            if (i // chunk_size + 1) % 3 == 0:
                clear_memory()
        
        return chunks, temp_dir, page_offsets
        
    except Exception as e:
        print(f"⚠️ Erreur division PDF : {e}")
        return [], None, []

# ============================================================================
# FONCTIONS DE FUSION AVEC PAGINATION CONTINUE
# ============================================================================

def renumber_pages_in_markdown(markdown_text, page_offset):
    """
    Renumérote les marqueurs de page dans le Markdown pour assurer une pagination continue.
    """
    import re
    
    if page_offset == 0:
        return markdown_text
    
    def replace_page_number(match):
        prefix = match.group(1)
        page_num = int(match.group(2))
        suffix = match.group(3) if len(match.groups()) > 2 else ""
        new_page = page_num + page_offset
        return f"{prefix}{new_page}{suffix}"
    
    markdown_text = re.sub(r'(\{page_?[nN]umber[:\s]*?)(\d+)(\})', replace_page_number, markdown_text)
    markdown_text = re.sub(r'(<!--\s*[Pp]age\s+)(\d+)(\s*-->)', replace_page_number, markdown_text)
    markdown_text = re.sub(r'([-=]{3,}\s*[Pp]age\s+)(\d+)(\s*[-=]{3,})', replace_page_number, markdown_text)
    markdown_text = re.sub(r'([\[(][Pp]age\s+)(\d+)([\])])', replace_page_number, markdown_text)
    markdown_text = re.sub(r'^(\s*[Pp]age\s+)(\d+)(\s*$)', replace_page_number, markdown_text, flags=re.MULTILINE)
    markdown_text = re.sub(r'(\n\n\{)(\d+)(\}\n\n)', replace_page_number, markdown_text)
    
    return markdown_text


def merge_markdowns(chunks_data, final_doc_name, page_offsets=None, chunk_size_used=CHUNK_SIZE_PAGES):
    """Fusionne les Markdowns des chunks avec séparateurs et pagination continue."""
    full_md = f"# {final_doc_name}\n\n"
    full_md += f"*Document fusionné le {datetime.now().strftime('%Y-%m-%d %H:%M')}*\n\n"
    
    total_pages = 0
    if page_offsets and chunks_data:
        last_chunk_pages = chunks_data[-1].get('ocr_analysis', {}).get('total_pages', chunk_size_used)
        if isinstance(last_chunk_pages, dict):
            last_chunk_pages = last_chunk_pages.get('total_pages', chunk_size_used)
        total_pages = page_offsets[-1] + last_chunk_pages if page_offsets else 0
        full_md += f"*Pages totales estimées: {total_pages}*\n\n"
    
    full_md += "---\n\n"
    
    for i, data in enumerate(chunks_data):
        chunk_md = ""
        if 'markdown_path' in data and data['markdown_path'] and Path(data['markdown_path']).exists():
            with open(data['markdown_path'], 'r', encoding='utf-8') as f:
                chunk_md = f.read()
        
        if page_offsets and i < len(page_offsets):
            offset = page_offsets[i]
            chunk_md = renumber_pages_in_markdown(chunk_md, offset)
            chunk_start_page = offset + 1
            chunk_end_page = page_offsets[i + 1] if i + 1 < len(page_offsets) else total_pages
            full_md += f"\n\n<!-- ═══════════════ CHUNK {i+1}/{len(chunks_data)} (Pages {chunk_start_page}-{chunk_end_page}) ═══════════════ -->\n\n"
        else:
            full_md += f"\n\n<!-- ═══════════════ CHUNK {i+1}/{len(chunks_data)} ═══════════════ -->\n\n"
        
        full_md += chunk_md
    
    full_md += "\n\n---\n*Fin du document*\n"
    
    doc_dirs = get_doc_dirs(final_doc_name)
    final_md_path = doc_dirs['root'] / f"{final_doc_name}.md"
    
    with open(final_md_path, 'w', encoding='utf-8') as f:
        f.write(full_md)
    
    if page_offsets:
        pagination_meta = {
            "total_pages": total_pages,
            "chunks_count": len(chunks_data),
            "page_offsets": page_offsets,
            "chunk_size_used": chunk_size_used,
            "merge_timestamp": datetime.now().isoformat()
        }
        meta_path = doc_dirs['logs'] / f"{final_doc_name}_pagination.json"
        with open(meta_path, 'w', encoding='utf-8') as f:
            json.dump(pagination_meta, f, indent=2)
    
    return str(final_md_path)


def merge_searchable_pdfs(chunks_pdfs, output_path):
    """Fusionne les PDFs Searchable en un seul document."""
    merger = PdfMerger()
    try:
        valid_pdfs = [p for p in chunks_pdfs if p and Path(p).exists()]
        if not valid_pdfs:
            print("   ⚠️ Aucun PDF valide à fusionner")
            return False
        
        print(f"   🔗 Fusion de {len(valid_pdfs)} PDFs...")
        for pdf in valid_pdfs:
            merger.append(str(pdf))
        
        merger.write(str(output_path))
        merger.close()
        print(f"   ✅ PDF fusionné : {output_path.name}")
        return True
        
    except Exception as e:
        print(f"   ⚠️ Erreur fusion PDF : {e}")
        return False
    finally:
        try:
            merger.close()
        except:
            pass


def merge_figures(chunks_data, final_doc_name):
    """Consolide les figures de tous les chunks avec renommage unique."""
    all_figures = []
    doc_dirs = get_doc_dirs(final_doc_name)
    final_figures_dir = doc_dirs['figures']
    
    for chunk_idx, res in enumerate(chunks_data):
        figures_paths = res.get('figures_paths', [])
        for fig_idx, fig_path in enumerate(figures_paths):
            if Path(fig_path).exists():
                new_name = f"chunk_{chunk_idx+1:02d}_fig_{fig_idx+1:02d}{Path(fig_path).suffix}"
                new_path = final_figures_dir / new_name
                try:
                    shutil.copy2(fig_path, new_path)
                    all_figures.append({
                        "original": str(fig_path),
                        "consolidated": str(new_path),
                        "chunk": chunk_idx + 1
                    })
                except Exception as e:
                    print(f"   ⚠️ Erreur copie figure : {e}")
    
    return all_figures


def deduplicate_references(chunks_data):
    """Déduplique les références bibliographiques des chunks."""
    all_refs = set()
    
    for res in chunks_data:
        refs = res.get('references', {}).get('references_list', [])
        for r in refs:
            normalized = r.strip().lower()
            if len(normalized) > 20:
                all_refs.add(r)
    
    return {
        "references_list": sorted(list(all_refs)),
        "reference_count": len(all_refs)
    }

# ============================================================================
# TRAITEMENT D'UN CHUNK INDIVIDUEL
# ============================================================================

def process_single_chunk(chunk_path, chunk_name, chunk_idx=0, total_chunks=1, page_offset=0, force_ocr_override=None):
    """Traite un chunk individuel : Marker → MD → OCR."""
    print(f"   ► Chunk {chunk_idx+1}/{total_chunks} ({chunk_name}) [Pages {page_offset+1}+]...")
    
    result_data = convert_pdf_complete(chunk_path, chunk_name, force_ocr_override=force_ocr_override)
    result_data['page_offset'] = page_offset
    
    if result_data.get('error'):
        print(f"     ❌ Erreur Marker : {result_data['error']}")
        return result_data
    
    chunk_dirs = get_doc_dirs(chunk_name)
    chunk_root = chunk_dirs['root']
    
    chunk_pdf_out = chunk_root / f"{chunk_name}_ocr.pdf"
    chunk_txt_sidecar = chunk_root / f"{chunk_name}_sidecar.txt"
    
    if result_data.get('markdown_path'):
        markdown_to_ocr_text(Path(result_data['markdown_path']), chunk_txt_sidecar)
    
    print(f"     🔍 OCR en cours...")
    if chunk_txt_sidecar.exists():
        success, err = make_searchable_pdf(chunk_path, chunk_pdf_out, sidecar_txt=chunk_txt_sidecar)
    else:
        success, err = make_searchable_pdf(chunk_path, chunk_pdf_out)
    
    if success:
        result_data['searchable_pdf'] = str(chunk_pdf_out)
        print(f"     ✅ Chunk traité")
    else:
        print(f"     ⚠️ OCR échoué : {err}")
        result_data['searchable_pdf'] = None
    
    clear_memory()
    return result_data

# ============================================================================
# CHUNK ADAPTATIF
# ============================================================================

def get_adaptive_chunk_size(ocr_analysis):
    """Détermine la taille de chunk optimale basée sur l'analyse OCR."""
    if ocr_analysis is None:
        return CHUNK_SIZE_PAGES
    pdf_type = ocr_analysis.get("pdf_type", "unknown")
    confidence = ocr_analysis.get("confidence", 0.0)
    
    if pdf_type == "native" and confidence >= NATIVE_CONFIDENCE_THRESHOLD:
        print(f"   📊 PDF natif (confiance {confidence:.0%}) → Chunks de {CHUNK_SIZE_PAGES_NATIVE} pages")
        return CHUNK_SIZE_PAGES_NATIVE
    
    print(f"   📊 PDF {pdf_type} (confiance {confidence:.0%}) → Chunks de {CHUNK_SIZE_PAGES} pages")
    return CHUNK_SIZE_PAGES


# ============================================================================
# VRAM MONITORING UTILITIES
# ============================================================================

VRAM_WARNING_THRESHOLD_GB = 12.0   # Seuil d'alerte VRAM (sur 15GB T4)
VRAM_CRITICAL_THRESHOLD_GB = 13.5  # Seuil critique → nettoyage d'urgence

def get_vram_usage_gb():
    """Retourne la VRAM réservée en GB."""
    if torch.cuda.is_available():
        return torch.cuda.memory_reserved(0) / (1024**3)
    return 0.0

def get_vram_allocated_gb():
    """Retourne la VRAM allouée (active) en GB."""
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated(0) / (1024**3)
    return 0.0

def vram_status_str():
    """Retourne un string lisible de l'état VRAM."""
    if not torch.cuda.is_available():
        return "CPU mode"
    alloc = get_vram_allocated_gb()
    reserved = get_vram_usage_gb()
    total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    return f"VRAM: {alloc:.1f}GB alloc / {reserved:.1f}GB reserved / {total:.1f}GB total"

def emergency_vram_cleanup():
    """Nettoyage d'urgence VRAM — appelé quand on approche du seuil critique."""
    print(f"   🚨 VRAM cleanup d'urgence ({vram_status_str()})")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    gc.collect()
    print(f"   → Après cleanup: {vram_status_str()}")

def check_vram_before_chunk(chunk_idx, total_chunks):
    """Vérifie la VRAM avant de traiter un chunk. Retourne True si OK."""
    vram = get_vram_usage_gb()
    if vram > VRAM_CRITICAL_THRESHOLD_GB:
        print(f"   ⚠️ VRAM critique ({vram:.1f}GB) avant chunk {chunk_idx+1}/{total_chunks}")
        emergency_vram_cleanup()
        vram = get_vram_usage_gb()
        if vram > VRAM_CRITICAL_THRESHOLD_GB:
            print(f"   🚨 VRAM toujours critique ({vram:.1f}GB) — chunk risqué")
            return False
    elif vram > VRAM_WARNING_THRESHOLD_GB:
        print(f"   ⚠️ VRAM élevée ({vram:.1f}GB) — nettoyage préventif")
        clear_memory()
    return True

print("✅ Monitoring VRAM chargé.")

# ============================================================================
# ORCHESTRATEUR PRINCIPAL POUR GROS PDFs — TRAITEMENT SÉQUENTIEL GPU-SAFE
# ============================================================================
# IMPORTANT : ThreadPoolExecutor a été SUPPRIMÉ.
# Raison : chaque chunk appelle converter() → Surya OCR sur GPU.
# Sur T4 (15GB VRAM), 2+ chunks simultanés = OOM = kernel restart silencieux.
# Le traitement séquentiel avec cleanup entre chunks est plus lent mais stable.
# ============================================================================

def process_large_pdf(pdf_path, doc_name, chunk_size=None, ocr_analysis=None):
    """
    Orchestre le découpage, traitement et fusion d'un gros PDF avec pagination continue.
    Traitement SÉQUENTIEL des chunks pour éviter les OOM GPU.
    force_ocr est déterminé par l'analyse OCR du PDF original.
    """
    print(f"\n🐘 TRAITEMENT GROS PDF : {doc_name}")
    print("=" * 50)
    print(f"   💾 VRAM initiale: {vram_status_str()}")
    
    start_time = time.time()
    
    # DÉTERMINATION ADAPTATIVE DE LA TAILLE DE CHUNK
    if chunk_size is None:
        chunk_size = get_adaptive_chunk_size(ocr_analysis)
    else:
        print(f"   📊 Taille de chunk forcée : {chunk_size} pages")
    
    # DÉTERMINATION force_ocr POUR TOUS LES CHUNKS (basé sur analyse du PDF complet)
    force_ocr_for_chunks = None  # None = détection adaptative par chunk
    if ocr_analysis is not None:
        if ocr_analysis.get("pdf_type") == "native" and ocr_analysis.get("confidence", 0) >= NATIVE_CONFIDENCE_THRESHOLD:
            force_ocr_for_chunks = False
            print(f"   📊 PDF natif détecté (confiance {ocr_analysis.get('confidence', 0):.0%}) → force_ocr=False pour tous les chunks")
        elif ocr_analysis.get("needs_ocr", True):
            force_ocr_for_chunks = True
            print(f"   📊 Type {ocr_analysis.get('pdf_type', 'unknown')} → force_ocr=True pour tous les chunks")
    
    # ÉTAPE 1 : DIVISION
    print("\n📂 [1/4] Division du PDF...")
    chunks, temp_dir, page_offsets = split_pdf(pdf_path, chunk_size)
    
    if not chunks:
        return {"doc_name": doc_name, "error": "Échec de la division du PDF", "chunked": True}
    
    print(f"   📑 Offsets de pagination: {page_offsets}")
    
    # ÉTAPE 2 : TRAITEMENT SÉQUENTIEL DES CHUNKS (GPU-safe)
    print(f"\n⚙️  [2/4] Traitement SÉQUENTIEL de {len(chunks)} chunks (GPU-safe)...")
    
    chunk_results = []
    chunk_pdfs = []
    processing_errors = []
    
    for idx, chunk_path in enumerate(chunks):
        offset = page_offsets[idx] if idx < len(page_offsets) else 0
        
                chunk_path, chunk_path.stem, idx, len(chunks), page_offset=offset,
                force_ocr_override=force_ocr_for_chunks
            )
            chunk_results.append(res)
            chunk_pdfs.append(res.get('searchable_pdf'))
            
            if res.get('error'):
                processing_errors.append(f"Chunk {idx+1}: {res['error']}")
            
        except Exception as e:
            print(f"     🚨 Exception Chunk {idx+1}: {e}")
            chunk_results.append({"error": str(e), "page_offset": offset})
            chunk_pdfs.append(None)
            processing_errors.append(f"Chunk {idx+1}: {str(e)}")
        
        # Nettoyage agressif entre les chunks
        clear_memory()
        print(f"     💾 Post-chunk {idx+1}: {vram_status_str()}")
    
    # ÉTAPE 3 : FUSION (avec pagination continue)
    print(f"\n🔗 [3/4] Fusion des résultats avec pagination continue...")
    
    merge_success = False
    final_md_path = None
    final_pdf_path = None
    all_figures = []
    references = {"references_list": [], "reference_count": 0}
    
    if any(r.get('markdown_path') for r in chunk_results):
        try:
            doc_dirs = get_doc_dirs(doc_name)
            
            final_md_path = merge_markdowns(
                chunk_results, doc_name, page_offsets=page_offsets, chunk_size_used=chunk_size
            )
            print(f"   ✅ Markdown fusionné avec pagination continue: {Path(final_md_path).name}")
            
            final_pdf_path = doc_dirs['root'] / f"{doc_name}_searchable.pdf"
            valid_pdfs = [p for p in chunk_pdfs if p and Path(p).exists()]
            
            if len(valid_pdfs) > 1:
                merge_success = merge_searchable_pdfs(valid_pdfs, final_pdf_path)
            elif len(valid_pdfs) == 1:
                print(f"   📋 Un seul PDF OCR - copie directe...")
                try:
                    shutil.copy2(valid_pdfs[0], final_pdf_path)
                    merge_success = True
                    print(f"   ✅ PDF copié : {final_pdf_path.name}")
                except Exception as e:
                    print(f"   ⚠️ Erreur copie PDF : {e}")
                    merge_success = False
            else:
                print(f"   ⚠️ Aucun PDF OCR valide à fusionner ({len(chunk_pdfs)} chunks traités)")
                merge_success = False
            
            all_figures = merge_figures(chunk_results, doc_name)
            if all_figures:
                print(f"   ✅ {len(all_figures)} figures consolidées")
            
            references = deduplicate_references(chunk_results)
            if references['reference_count'] > 0:
                print(f"   ✅ {references['reference_count']} références dédupliquées")
            
        except Exception as e:
            print(f"   ⚠️ Erreur fusion : {e}")
            processing_errors.append(f"Fusion: {str(e)}")
    
    # ÉTAPE 4 : NETTOYAGE
    print(f"\n🧹 [4/4] Nettoyage...")
    
    should_cleanup = final_md_path is not None and Path(final_md_path).exists()
    
    if should_cleanup:
        cleanup_errors = []
        
        try:
            if temp_dir and temp_dir.exists():
                shutil.rmtree(temp_dir)
                print(f"   ✅ Dossier temp supprimé: {temp_dir.name}")
        except Exception as e:
            cleanup_errors.append(f"temp_dir: {e}")
        
        chunks_cleaned = 0
        for res in chunk_results:
            chunk_doc_name = res.get('doc_name')
            if chunk_doc_name and "_chunk_" in chunk_doc_name:
                chunk_dir = OUTPUT_DIR / chunk_doc_name
                if chunk_dir.exists() and chunk_dir.is_dir():
                    try:
                        shutil.rmtree(chunk_dir)
                        chunks_cleaned += 1
                    except Exception as e:
                        cleanup_errors.append(f"{chunk_doc_name}: {e}")
        
        if chunks_cleaned > 0:
            print(f"   ✅ {chunks_cleaned} dossiers de chunks supprimés")
        
        if cleanup_errors:
            print(f"   ⚠️ Nettoyage partiel ({len(cleanup_errors)} erreurs)")
            for err in cleanup_errors[:3]:
                print(f"      - {err}")
    else:
        print(f"   ⚠️ Fusion échouée - Chunks conservés pour debug: {temp_dir}")
        print(f"      → Vérifiez manuellement les erreurs et relancez si nécessaire")
    
    duration = time.time() - start_time
    print(f"\n{'='*50}")
    print(f"✅ Terminé en {duration:.1f}s")
    
    total_pages = page_offsets[-1] + chunk_size if page_offsets else 0
    
    return {
        "doc_name": doc_name,
        "markdown_path": str(final_md_path) if final_md_path else None,
        "searchable_pdf": str(final_pdf_path) if final_pdf_path else None,
        "figures": all_figures,
        "references": references,
        "error": "; ".join(processing_errors) if processing_errors else None,
        "chunked": True,
        "chunks_count": len(chunks),
        "chunk_size_used": chunk_size,
        "page_offsets": page_offsets,
        "total_pages_estimated": total_pages,
        "duration": duration
    }


print("✅ Fonctions de gestion des gros PDFs chargées (chunk unifié=50 pages, traitement SÉQUENTIEL GPU-safe).")
        "references": references,print("✅ Fonctions de gestion des gros PDFs chargées (chunk unifié=50 pages, traitement SÉQUENTIEL GPU-safe).")

        "error": "; ".join(processing_errors) if processing_errors else None,

        "chunked": True,    }

        "chunks_count": len(chunks),        "duration": duration

        "chunk_size_used": chunk_size,        "total_pages_estimated": total_pages,
        "page_offsets": page_offsets,

## Étape 8b — Test sur un PDF avec monitoring VRAM (optionnel)

Permet de valider la configuration et les fonctions de gestion des gros PDFs avant le lancement du batch complet.

**Ce test vérifie :**
- ✅ Détection automatique de la taille/complexité
- ✅ Basculement mode Chunking si nécessaire  
- ✅ Pipeline complet Marker → MD → OCR → PDF Searchable
- ✅ **Monitoring VRAM** avant/pendant/après le traitement
- ✅ **Traitement séquentiel** des chunks (pas de crash kernel)

In [ ]:
# ============================================================================
# TEST SUR UN PDF AVEC MONITORING VRAM
# ============================================================================
if pdf_files:
    sample_path = pdf_files[0]
    sample_name = sample_path.stem
    
    print(f"🧪 Test de conversion sur : {sample_name}")
    print(f"📊 Taille : {sample_path.stat().st_size / (1024*1024):.1f} MB")
    print(f"📄 Pages : {get_pdf_page_count(sample_path)}")
    print(f"💾 VRAM avant : {vram_status_str()}")

    # Vérification santé et choix de la méthode
    is_large, warnings, ocr_analysis = check_pdf_health(sample_path, max_size_mb=MAX_PDF_SIZE_MB, max_pages=MAX_PDF_PAGES)
    
    if is_large:
        print(f"\n⚠️ PDF volumineux détecté ({', '.join(warnings)}).")
        print("⚡ Passage automatique en mode 'Chunking SÉQUENTIEL' (GPU-safe).")
        # Utilise le chunk unifié (50 pages pour tous types)
        sample_result = process_large_pdf(sample_path, sample_name, ocr_analysis=ocr_analysis)
    else:
        print("\n✅ PDF de taille standard. Traitement direct.")
        try:
            # Creation structure dossier
            doc_dirs = get_doc_dirs(sample_name)
            
            print(f"💾 VRAM avant Marker : {vram_status_str()}")
            sample_result = convert_pdf_complete(sample_path, sample_name)
            print(f"💾 VRAM après Marker : {vram_status_str()}")
            
            # OCR pour mode standard
            if not sample_result.get("error"):
                out_pdf = doc_dirs['root'] / f"{sample_name}_searchable.pdf"
                sidecar_txt = doc_dirs['root'] / f"{sample_name}_sidecar.txt"
                
                # MD → TXT pour guider l'OCR
                if sample_result.get('markdown_path'):
                    markdown_to_ocr_text(Path(sample_result['markdown_path']), sidecar_txt)
                
                # OCRmyPDF
                print(f"💾 VRAM avant OCR : {vram_status_str()}")
                ok, err = make_searchable_pdf(sample_path, out_pdf, sidecar_txt=sidecar_txt if sidecar_txt.exists() else None)
                if ok:
                    sample_result["searchable_pdf"] = str(out_pdf)
                else:
                    print(f"⚠️ Erreur OCR : {err}")
                    
        except Exception as e:
            if "memory" in str(e).lower():
                print("🚨 OOM détecté en mode standard. Tentative de reprise en mode Chunking...")
                clear_memory()
                sample_result = process_large_pdf(sample_path, sample_name, ocr_analysis=ocr_analysis)
            else:
                sample_result = {"error": str(e)}

    # Nettoyage post-test
    clear_memory()

    # Affichage résultat
    print("\n" + "="*50)
    print("📋 RÉSULTAT DU TEST")
    print("="*50)
    
    if sample_result.get("error"):
        print(f"❌ Erreur : {sample_result['error']}")
    else:
        print(f"✅ Succès !")
        print(f"📄 Markdown : {sample_result.get('markdown_path')}")
        print(f"🔍 PDF OCR  : {sample_result.get('searchable_pdf')}")
        print(f"🖼️  Figures  : {len(sample_result.get('figures', []))}")
        print(f"📚 Refs     : {sample_result.get('references', {}).get('reference_count', 0)}")
        
        if sample_result.get('chunked'):
            print(f"✂️  Chunks   : {sample_result.get('chunks_count', 'N/A')}")
            print(f"⏱️  Durée    : {sample_result.get('duration', 0):.1f}s")
    
    print(f"\n💾 VRAM finale : {vram_status_str()}")
else:
    print("⚠️ Aucun PDF trouvé dans le dossier d'entrée.")

## Étape 9 — Pipeline batch avec reprise automatique

Traitement batch intelligent avec monitoring et reprise après crash.

**Fonctionnalités :**
- 📊 Tri des PDFs par complexité (les plus petits d'abord)
- 🐘 Détection automatique des gros PDFs → mode Chunking
- 💾 Checkpoints tous les 5 documents
- 🔄 Synchronisation incrémentale vers Drive
- 📈 Dashboard temps réel avec progression
- 📋 Rapport CSV final avec métriques

In [ ]:
# ============================================================================
# DASHBOARD TEMPS RÉEL
# ============================================================================
from IPython.display import display, HTML, clear_output
import time
import pandas as pd

class ProgressDashboard:
    """Dashboard de progression temps réel pour le pipeline."""
    
    def __init__(self, total_files):
        self.total = total_files
        self.processed = 0
        self.success = 0
        self.errors = 0
        self.start_time = time.time()
        self.current_file = ""
        self.results = []
        
    def update(self, doc_name, status, duration, error=None):
        """Met à jour le dashboard après traitement d'un fichier."""
        self.processed += 1
        self.current_file = doc_name
        
        if status == "SUCCESS":
            self.success += 1
        else:
            self.errors += 1
        
        self.results.append({
            "doc": doc_name,
            "status": status,
            "duration": f"{duration:.1f}s",
            "error": error or ""
        })
        
    def display(self):
        """Affiche le dashboard HTML."""
        elapsed = time.time() - self.start_time
        
        # Calculs
        progress_pct = (self.processed / self.total * 100) if self.total > 0 else 0
        success_rate = (self.success / self.processed * 100) if self.processed > 0 else 0
        avg_time = elapsed / self.processed if self.processed > 0 else 0
        remaining = self.total - self.processed
        eta_seconds = remaining * avg_time
        eta_str = f"{int(eta_seconds // 60)}m {int(eta_seconds % 60)}s" if eta_seconds > 0 else "—"
        
        # Couleur barre de progression
        bar_color = "#4CAF50" if success_rate >= 80 else "#FF9800" if success_rate >= 50 else "#F44336"
        
        # HTML du dashboard
        html = f"""
        <div style="font-family: Arial, sans-serif; padding: 15px; background: #1e1e1e; border-radius: 10px; color: #fff;">
            <h3 style="margin: 0 0 15px 0; color: #4FC3F7;">📊 Pipeline Progress</h3>
            
            <!-- Barre de progression -->
            <div style="background: #333; border-radius: 10px; height: 30px; margin-bottom: 15px; overflow: hidden;">
                <div style="background: {bar_color}; height: 100%; width: {progress_pct}%; 
                            display: flex; align-items: center; justify-content: center;
                            font-weight: bold; transition: width 0.3s;">
                    {progress_pct:.1f}%
                </div>
            </div>
            
            <!-- Métriques principales -->
            <div style="display: flex; gap: 20px; margin-bottom: 15px;">
                <div style="flex: 1; background: #2d2d2d; padding: 10px; border-radius: 8px; text-align: center;">
                    <div style="font-size: 24px; font-weight: bold; color: #4FC3F7;">{self.processed}/{self.total}</div>
                    <div style="font-size: 12px; color: #888;">Traités</div>
                </div>
                <div style="flex: 1; background: #2d2d2d; padding: 10px; border-radius: 8px; text-align: center;">
                    <div style="font-size: 24px; font-weight: bold; color: #4CAF50;">{self.success}</div>
                    <div style="font-size: 12px; color: #888;">Réussis</div>
                </div>
                <div style="flex: 1; background: #2d2d2d; padding: 10px; border-radius: 8px; text-align: center;">
                    <div style="font-size: 24px; font-weight: bold; color: #F44336;">{self.errors}</div>
                    <div style="font-size: 12px; color: #888;">Erreurs</div>
                </div>
                <div style="flex: 1; background: #2d2d2d; padding: 10px; border-radius: 8px; text-align: center;">
                    <div style="font-size: 24px; font-weight: bold; color: #FF9800;">{success_rate:.0f}%</div>
                    <div style="font-size: 12px; color: #888;">Taux réussite</div>
                </div>
            </div>
            
            <!-- Temps et vitesse -->
            <div style="display: flex; gap: 20px; margin-bottom: 15px;">
                <div style="flex: 1; background: #2d2d2d; padding: 10px; border-radius: 8px;">
                    <span style="color: #888;">⏱️ Écoulé:</span> 
                    <span style="font-weight: bold;">{int(elapsed // 60)}m {int(elapsed % 60)}s</span>
                </div>
                <div style="flex: 1; background: #2d2d2d; padding: 10px; border-radius: 8px;">
                    <span style="color: #888;">⏳ ETA:</span> 
                    <span style="font-weight: bold;">{eta_str}</span>
                </div>
                <div style="flex: 1; background: #2d2d2d; padding: 10px; border-radius: 8px;">
                    <span style="color: #888;">🚀 Vitesse:</span> 
                    <span style="font-weight: bold;">{avg_time:.1f}s/doc</span>
                </div>
            </div>
            
            <!-- Fichier en cours -->
            <div style="background: #2d2d2d; padding: 10px; border-radius: 8px; margin-bottom: 10px;">
                <span style="color: #888;">📄 En cours:</span> 
                <span style="font-weight: bold; color: #4FC3F7;">{self.current_file or "—"}</span>
            </div>
            
            <!-- Mémoire -->
            <div style="background: #2d2d2d; padding: 10px; border-radius: 8px;">
                <span style="color: #888;">💾 Mémoire:</span> 
                <span style="font-weight: bold;">{get_memory_stats()}</span>
            </div>
        </div>
        """
        
        clear_output(wait=True)
        display(HTML(html))
        
    def get_summary_df(self):
        """Retourne un DataFrame résumé."""
        return pd.DataFrame(self.results)


def display_final_report(results_list, start_time):
    """Affiche le rapport final enrichi."""
    elapsed = time.time() - start_time
    df = pd.DataFrame(results_list)
    
    success_count = len(df[df['status'] == 'SUCCESS'])
    error_count = len(df[df['status'] != 'SUCCESS'])
    
    html = f"""
    <div style="font-family: Arial, sans-serif; padding: 20px; background: #1e1e1e; border-radius: 10px; color: #fff;">
        <h2 style="color: #4CAF50; margin-bottom: 20px;">🏁 Rapport Final</h2>
        
        <div style="display: flex; gap: 20px; margin-bottom: 20px;">
            <div style="flex: 1; background: #4CAF50; padding: 20px; border-radius: 10px; text-align: center;">
                <div style="font-size: 36px; font-weight: bold;">{success_count}</div>
                <div>Réussis</div>
            </div>
            <div style="flex: 1; background: #F44336; padding: 20px; border-radius: 10px; text-align: center;">
                <div style="font-size: 36px; font-weight: bold;">{error_count}</div>
                <div>Erreurs</div>
            </div>
            <div style="flex: 1; background: #2196F3; padding: 20px; border-radius: 10px; text-align: center;">
                <div style="font-size: 36px; font-weight: bold;">{elapsed/60:.1f}m</div>
                <div>Durée totale</div>
            </div>
        </div>
        
        <h3 style="color: #888;">📋 Détails par document</h3>
    </div>
    """
    
    display(HTML(html))
    display(df)

print("✅ Dashboard de progression chargé.")

### Exécution du Pipeline

Lancez la cellule suivante pour démarrer le traitement batch complet. Le dashboard s'affichera automatiquement et se mettra à jour en temps réel.

In [ ]:
# ============================================================================
# PIPELINE BATCH AVEC REPRISE AUTOMATIQUE ET SYNC VÉRIFIÉ
# ============================================================================
# Création d'un dossier de logs global pour le rapport général
GLOBAL_LOGS_DIR = OUTPUT_DIR / "_GLOBAL_LOGS"
GLOBAL_LOGS_DIR.mkdir(parents=True, exist_ok=True)

# Configuration des fichiers de log et checkpoint
report_file = GLOBAL_LOGS_DIR / f"report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
checkpoint_file = GLOBAL_LOGS_DIR / "checkpoint.json"

# ═══════════════════════════════════════════════════════════════════════════
# 1. CHARGEMENT DU CHECKPOINT (REPRISE)
# ═══════════════════════════════════════════════════════════════════════════
processed_files = set()
if checkpoint_file.exists():
    try:
        with open(checkpoint_file, "r") as f:
            processed_files = set(json.load(f))
        print(f"🔄 Reprise locale détectée : {len(processed_files)} fichiers déjà logs.")
    except:
        print("⚠️ Checkpoint corrompu, ignoré.")

# --- NOUVEAU : Vérification des fichiers déjà existants sur Drive (Output) ---
if DRIVE_OUTPUT_DIR.exists():
    print(f"🔍 Scan des documents existants sur Drive ({DRIVE_OUTPUT_DIR})...")
    try:
        drive_existing_count = 0
        # On suppose que chaque sous-dossier dans OUTPUT est un document traité
        for item in DRIVE_OUTPUT_DIR.iterdir():
            if item.is_dir() and not item.name.startswith('_'): # Ignorer _GLOBAL_LOGS, etc.
                if item.name not in processed_files:
                    processed_files.add(item.name)
                    drive_existing_count += 1
        
        if drive_existing_count > 0:
            print(f"   ☁️  {drive_existing_count} documents supplémentaires trouvés sur Drive (seront ignorés).")
        
        print(f"   ✅ Total documents à ignorer (Déjà traités) : {len(processed_files)}")
    except Exception as e:
        print(f"   ⚠️ Erreur lors de la lecture du Drive (reprise Drive impossible) : {e}")
else:
    print(f"⚠️ Dossier Drive Output introuvable : reprise basée uniquement sur fichiers locaux.")

# ═══════════════════════════════════════════════════════════════════════════
# 2. ANALYSE ET TRI INTELLIGENT DES FICHIERS
# ═══════════════════════════════════════════════════════════════════════════
print("\n📊 Analyse et tri des documents...")
pdf_files_ranked = []
large_files_warning = []
ocr_stats = {"native": 0, "scan": 0, "mixed": 0, "bad_ocr": 0}

for p in pdf_files:
    complexity = get_pdf_complexity(p)
    is_large, warnings, ocr_analysis = check_pdf_health(p, max_size_mb=MAX_PDF_SIZE_MB, max_pages=MAX_PDF_PAGES)
    pdf_files_ranked.append((p, complexity, is_large, ocr_analysis))
    
    if is_large:
        large_files_warning.append((p.name, warnings))
    
    # Stats OCR
    if ocr_analysis:
        pdf_type = ocr_analysis.get("pdf_type", "unknown")
        if pdf_type in ocr_stats:
            ocr_stats[pdf_type] += 1

# Tri par complexité croissante
pdf_files_ranked.sort(key=lambda x: x[1])

# Afficher les stats OCR (informatif — scénario ADAPTATIF)
print(f"\n📋 Analyse des documents (scénario ADAPTATIF) :")
print(f"   - PDF natifs (texte propre) : {ocr_stats['native']} → force_ocr=False (économie T4)")
print(f"   - Scans (images)            : {ocr_stats['scan']} → force_ocr=True (Surya OCR)")
print(f"   - OCR existant bruité       : {ocr_stats['bad_ocr']} → force_ocr=True (Surya OCR)")
print(f"   - Mixtes/incertains         : {ocr_stats['mixed']} → force_ocr=True (Surya OCR)")
print(f"   🎯 Scénario actif : {OCR_SCENARIO}")

# Afficher les extrêmes et warnings
if pdf_files_ranked:
    print(f"   📉 Moins complexe : {pdf_files_ranked[0][0].name} (Score: {pdf_files_ranked[0][1]:.0f})")
    print(f"   📈 Plus complexe  : {pdf_files_ranked[-1][0].name} (Score: {pdf_files_ranked[-1][1]:.0f})")
    
if large_files_warning:
    print(f"\n⚠️  {len(large_files_warning)} PDFs nécessiteront le mode Chunking :")
    for name, warns in large_files_warning[:5]:  # Afficher max 5
        print(f"   - {name}: {', '.join(warns)}")
    if len(large_files_warning) > 5:
        print(f"   ... et {len(large_files_warning) - 5} autres")

# Liste finale à traiter (inclut maintenant ocr_analysis)
pdf_queue = [(p, is_large, ocr_analysis) for p, _, is_large, ocr_analysis in pdf_files_ranked if p.stem not in processed_files]
print(f"\n🚀 {len(pdf_queue)} documents à traiter (sur {len(pdf_files_ranked)} total)")

# ═══════════════════════════════════════════════════════════════════════════
# 3. INITIALISATION DU PIPELINE
# ═══════════════════════════════════════════════════════════════════════════
results_list = []
start_time = time.time()
CHECKPOINT_INTERVAL = 5
SYNC_VERIFICATION_INTERVAL = 10  # Vérification complète tous les 10 documents
DASHBOARD_UPDATE_INTERVAL = 1

# Tracking des syncs
sync_stats = {
    "synced_ok": 0,
    "sync_failed": [],
    "last_verification": None
}

dashboard = ProgressDashboard(len(pdf_queue))
print(f"💾 Mémoire initiale : {get_memory_stats()}")
print(f"💾 VRAM initiale : {vram_status_str()}")

# ═══════════════════════════════════════════════════════════════════════════
# 4. BOUCLE PRINCIPALE DE TRAITEMENT
# ═══════════════════════════════════════════════════════════════════════════

for i, (pdf_path, is_large, ocr_analysis) in enumerate(pdf_queue):
    doc_name = pdf_path.stem
    iter_start = time.time()
    dashboard.current_file = doc_name
    
    clear_memory()
    
    try:
        # ADAPTATIF : force_ocr déterminé par l'analyse OCR du PDF
        # PDFs natifs → force_ocr=False (pas de Surya = économie VRAM T4)
        # Scans / bad_ocr → force_ocr=True (Surya OCR nécessaire)
        if ocr_analysis is not None:
            force_ocr_for_doc = ocr_analysis.get("needs_ocr", True)
            if ocr_analysis.get("pdf_type") == "native" and ocr_analysis.get("confidence", 0) >= NATIVE_CONFIDENCE_THRESHOLD:
                force_ocr_for_doc = False
        else:
            force_ocr_for_doc = None  # Détection adaptative dans convert_pdf_complete
    
        # ─────────────────────────────────────────────────────────────────
        # A. TRAITEMENT ADAPTATIF (chunk unifié: 50 pages)
        # ─────────────────────────────────────────────────────────────────
        if is_large:
            result_data = process_large_pdf(pdf_path, doc_name, ocr_analysis=ocr_analysis)
        else:
            result_data = convert_pdf_complete(pdf_path, doc_name, force_ocr_override=force_ocr_for_doc)
            
            # FIX: doc_dirs must be defined here (convert_pdf_complete creates it internally
            # but doesn't expose it to this scope)
            doc_dirs = get_doc_dirs(doc_name)
            
            if not result_data.get("error"):
                out_pdf = doc_dirs['root'] / f"{doc_name}_searchable.pdf"
                sidecar_txt = doc_dirs['root'] / f"{doc_name}_sidecar.txt"
                
                if result_data.get('markdown_path'):
                    markdown_to_ocr_text(Path(result_data['markdown_path']), sidecar_txt)
                
                ok, err = make_searchable_pdf(pdf_path, out_pdf, sidecar_txt=sidecar_txt if sidecar_txt.exists() else None)
                if ok:
                    result_data["searchable_pdf"] = str(out_pdf)

        # ─────────────────────────────────────────────────────────────────
        # B. ENRICHISSEMENT LANGEXTRACT (OPTIONNEL)
        # ─────────────────────────────────────────────────────────────────
        if not result_data.get("error") and USE_LANGEXTRACT:
            md_content = ""
            if result_data.get("markdown_path") and Path(result_data["markdown_path"]).exists():
                with open(result_data["markdown_path"], "r", encoding="utf-8") as f:
                    md_content = f.read()
            
            extraction = extract_with_langextract(
                md_content, doc_name,
                result_data.get("references"),
                result_data.get("figures")
            )
            
            if extraction.get("status") != "skipped":
                # doc_dirs already defined above for non-large PDFs
                if not is_large:
                    json_path = doc_dirs['analyses'] / f"{doc_name}_structure.json"
                else:
                    langext_dirs = get_doc_dirs(doc_name)
                    json_path = langext_dirs['analyses'] / f"{doc_name}_structure.json"
                with open(json_path, "w", encoding="utf-8") as f:
                    json.dump(extraction, f, ensure_ascii=False, indent=2)
                result_data["extraction_json"] = str(json_path)

        # ─────────────────────────────────────────────────────────────────
        # C. SYNC IMMÉDIAT AVEC VÉRIFICATION
        # ─────────────────────────────────────────────────────────────────
        sync_success = False
        sync_message = ""
        
        # Sync si le dossier du document existe et contient des fichiers
        doc_local_path = OUTPUT_DIR / doc_name
        has_output = doc_local_path.exists() and any(doc_local_path.rglob("*"))
        
        if has_output:
            sync_success, sync_message = sync_to_drive_with_verification(
                OUTPUT_DIR, DRIVE_OUTPUT_DIR, doc_name=doc_name
            )
            
            if sync_success:
                sync_stats["synced_ok"] += 1
            else:
                sync_stats["sync_failed"].append({
                    "doc": doc_name,
                    "message": sync_message,
                    "timestamp": datetime.now().isoformat()
                })
                print(f"   ⚠️ Sync échoué pour {doc_name}: {sync_message}")
        else:
            print(f"   ⚠️ Aucun fichier à synchroniser pour {doc_name}")

        # ─────────────────────────────────────────────────────────────────
        # D. LOGGING (avec statut sync et diagnostic)
        # ─────────────────────────────────────────────────────────────────
        duration = time.time() - iter_start
        status = "SUCCESS" if not result_data.get("error") else "ERROR"
        
        # Diagnostic des fichiers générés
        files_created = {
            "markdown": result_data.get("markdown_path") and Path(result_data.get("markdown_path")).exists(),
            "pdf_ocr": result_data.get("searchable_pdf") and Path(result_data.get("searchable_pdf")).exists(),
            "synced": sync_success
        }
        
        log_entry = {
            "doc_name": doc_name,
            "status": status,
            "duration": round(duration, 2),
            "chunked": result_data.get("chunked", False),
            "chunk_size": result_data.get("chunk_size_used", "N/A"),
            "figures_count": len(result_data.get("figures", [])),
            "refs_count": result_data.get("references", {}).get("reference_count", 0),
            "has_markdown": files_created["markdown"],
            "has_pdf_ocr": files_created["pdf_ocr"],
            "sync_status": "OK" if sync_success else "FAILED",
            "error": result_data.get("error"),
            "path": str(pdf_path)
        }
        results_list.append(log_entry)
        
        # Afficher un résumé si erreur
        if result_data.get("error"):
            print(f"   📋 Résumé: MD={'✓' if files_created['markdown'] else '✗'} | PDF={'✓' if files_created['pdf_ocr'] else '✗'} | Sync={'✓' if sync_success else '✗'}")
        
        dashboard.update(doc_name, status, duration, result_data.get("error"))
        dashboard.display()

        # ─────────────────────────────────────────────────────────────────
        # E. CHECKPOINT & VÉRIFICATION PÉRIODIQUE
        # ─────────────────────────────────────────────────────────────────
        processed_files.add(doc_name)
        
        # Checkpoint CHAQUE document — protection contre kernel crash
        pd.DataFrame(results_list).to_csv(report_file, index=False)
        with open(checkpoint_file, "w") as f:
            json.dump(list(processed_files), f)
        
        # Vérification périodique de la sync (tous les N documents)
        if (i + 1) % SYNC_VERIFICATION_INTERVAL == 0:
            print(f"\n🔍 Vérification périodique sync ({i+1}/{len(pdf_queue)})...")
            all_ok, verify_report = periodic_sync_verification(
                OUTPUT_DIR, DRIVE_OUTPUT_DIR, processed_files, sample_size=5
            )
            
            sync_stats["last_verification"] = verify_report
            
            if all_ok:
                print(f"   ✅ Vérification OK: {verify_report['verified']}/{verify_report['checked']} docs")
            else:
                print(f"   ⚠️ {len(verify_report['failed'])} docs à resync")
                resync_results = resync_failed_documents(
                    verify_report['failed'], OUTPUT_DIR, DRIVE_OUTPUT_DIR
                )
                for r in resync_results:
                    if r['success'] and r['doc'] in [f['doc'] for f in sync_stats['sync_failed']]:
                        sync_stats['sync_failed'] = [
                            f for f in sync_stats['sync_failed'] if f['doc'] != r['doc']
                        ]
                        sync_stats['synced_ok'] += 1
        
        # Sauvegarder les stats de sync périodiquement
        if (i + 1) % CHECKPOINT_INTERVAL == 0:
            sync_stats_file = GLOBAL_LOGS_DIR / "sync_stats.json"
            with open(sync_stats_file, "w") as f:
                json.dump(sync_stats, f, indent=2, default=str)

    except Exception as e:
        duration = time.time() - iter_start
        error_msg = str(e)
        
        results_list.append({
            "doc_name": doc_name,
            "status": "CRITICAL_ERROR",
            "duration": round(duration, 2),
            "sync_status": "N/A",
            "error": error_msg,
            "path": str(pdf_path)
        })
        
        # Sauvegarde d'urgence du checkpoint avant un éventuel crash kernel
        try:
            processed_files.add(doc_name)
            pd.DataFrame(results_list).to_csv(report_file, index=False)
            with open(checkpoint_file, "w") as f:
                json.dump(list(processed_files), f)
            print(f"   💾 Checkpoint d'urgence sauvegardé après erreur sur {doc_name}")
        except:
            pass  # Si même ça échoue, on ne peut rien faire
        
        # Nettoyage mémoire d'urgence
        clear_memory()
        if torch.cuda.is_available() and get_vram_usage_gb() > VRAM_CRITICAL_THRESHOLD_GB:
            emergency_vram_cleanup()

# ═══════════════════════════════════════════════════════════════════════════
# 5. VÉRIFICATION FINALE ET RESYNC
# ═══════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("🔍 VÉRIFICATION FINALE DE LA SYNCHRONISATION")
print("="*60)

# Vérification complète de tous les documents traités
all_ok, final_report = periodic_sync_verification(
    OUTPUT_DIR, DRIVE_OUTPUT_DIR, processed_files, sample_size=min(20, len(processed_files))
)

print(f"📊 Résultat: {final_report['verified']}/{final_report['checked']} documents vérifiés OK")

if final_report['failed']:
    print(f"\n⚠️ {len(final_report['failed'])} documents à resynchroniser:")
    resync_results = resync_failed_documents(final_report['failed'], OUTPUT_DIR, DRIVE_OUTPUT_DIR)
    
    still_failed = [r for r in resync_results if not r['success']]
    if still_failed:
        print(f"\n❌ {len(still_failed)} documents n'ont pas pu être synchronisés:")
        for r in still_failed:
            print(f"   - {r['doc']}: {r['message']}")

# Statistiques finales
print(f"\n📈 Statistiques Sync:")
print(f"   - Documents sync OK: {sync_stats['synced_ok']}")
print(f"   - Échecs restants: {len(sync_stats['sync_failed'])}")

print("\n✅ Pipeline terminé.")

# Sauvegarde finale

pd.DataFrame(results_list).to_csv(report_file, index=False)    json.dump(list(processed_files), f)
with open(checkpoint_file, "w") as f:

In [ ]:
# Synchronisation Finale
from pathlib import Path

# Définir les chemins si non définis (pour exécution indépendante)
if 'OUTPUT_DIR' not in dir():
    OUTPUT_DIR = Path("/content/work/ALLMD")
if 'DRIVE_OUTPUT_DIR' not in dir():
    DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/PHDM/ALL/ALLMD")

print("="*60)
print(f"🚀 SYNCHRONISATION FINALE VERS DRIVE")
print("="*60)
print(f"📂 Source :      {OUTPUT_DIR}")
print(f"☁️  Destination : {DRIVE_OUTPUT_DIR}")

# Utilise la fonction robuste définie précédemment
sync_to_drive(OUTPUT_DIR, DRIVE_OUTPUT_DIR)

# Résumé
try:
    md_count = len(list(DRIVE_OUTPUT_DIR.glob("**/*.md")))
    pdf_count = len(list(DRIVE_OUTPUT_DIR.glob("**/*.pdf")))
    print(f"\n📊 Bilan sur Drive :")
    print(f"   - Markdowns : {md_count}")
    print(f"   - PDFs      : {pdf_count}")
except:
    pass

print("✅ Traitement terminé.")

In [ ]:
# ============================================================================
# SYNCHRONISATION DRIVE ↔ GITHUB REPOSITORY
# ============================================================================
# Synchronise les dossiers ALL/ALLMD et ALL/ALLPDF entre Google Drive
# et une branche spécifique d'un repository GitHub.
#
# MODES :
#   - PUSH : Drive → GitHub (commit & push les fichiers vers le repo)
#   - PULL : GitHub → Drive (récupère les fichiers du repo vers Drive)
#
# ⚠️ Nécessite un token GitHub (Personal Access Token) avec scope "repo".
# ============================================================================

import os
import subprocess
import shutil
from pathlib import Path
from datetime import datetime

# ═══════════════════════════════════════════════════════════════════════════
# CONFIGURATION — MODIFIEZ CES VALEURS
# ═══════════════════════════════════════════════════════════════════════════

GITHUB_REPO = "https://github.com/tokyollover/PHDM.git"  # Repository
GITHUB_BRANCH = "PHDMWNB"                                  # Branche cible
SYNC_MODE = "PUSH"                                          # "PUSH" (Drive→GitHub) ou "PULL" (GitHub→Drive)

# Chemins Drive (source/destination selon le mode)
DRIVE_ALLMD = Path("/content/drive/MyDrive/PHDM/ALL/ALLMD")
DRIVE_ALLPDF = Path("/content/drive/MyDrive/PHDM/ALL/ALLPDF")

# Dossiers à synchroniser (relatifs à la racine du repo)
REPO_DIRS = ["ALL/ALLMD", "ALL/ALLPDF"]

# Dossier de travail temporaire pour le clone git
GIT_WORK_DIR = Path("/content/git_sync")

# ═══════════════════════════════════════════════════════════════════════════
# AUTHENTIFICATION GITHUB
# ═══════════════════════════════════════════════════════════════════════════

# Option 1 : Token via variable d'environnement (recommandé)
# Définissez GITHUB_TOKEN dans vos secrets Colab ou via os.environ
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "")

# Option 2 : Saisie interactive si pas de token
if not GITHUB_TOKEN:
    from getpass import getpass
    GITHUB_TOKEN = getpass("🔑 Entrez votre GitHub Personal Access Token (scope: repo) : ")

if not GITHUB_TOKEN:
    raise ValueError("❌ Token GitHub requis. Créez-en un sur https://github.com/settings/tokens")

# Construire l'URL authentifiée
REPO_URL_AUTH = GITHUB_REPO.replace("https://", f"https://{GITHUB_TOKEN}@")

# ═══════════════════════════════════════════════════════════════════════════
# FONCTIONS DE SYNCHRONISATION
# ═══════════════════════════════════════════════════════════════════════════

def run_cmd(cmd, cwd=None, check=True):
    """Exécute une commande shell avec gestion d'erreur."""
    result = subprocess.run(
        cmd, shell=True, cwd=cwd,
        capture_output=True, text=True, timeout=600
    )
    if check and result.returncode != 0:
        print(f"⚠️ Commande échouée: {cmd}")
        print(f"   STDERR: {result.stderr[:500]}")
        return False, result.stderr
    return True, result.stdout


def clone_or_update_repo(work_dir, repo_url, branch):
    """Clone le repo ou met à jour si déjà cloné."""
    repo_dir = work_dir / "PHDM"
    
    if repo_dir.exists() and (repo_dir / ".git").exists():
        print(f"   📂 Repo existant détecté, mise à jour...")
        run_cmd(f"git fetch origin", cwd=str(repo_dir))
        run_cmd(f"git checkout {branch}", cwd=str(repo_dir))
        run_cmd(f"git pull origin {branch}", cwd=str(repo_dir))
    else:
        print(f"   📥 Clonage du repo (branche: {branch})...")
        if repo_dir.exists():
            shutil.rmtree(repo_dir)
        work_dir.mkdir(parents=True, exist_ok=True)
        
        # Clone sparse (uniquement les dossiers nécessaires) pour économiser espace/temps
        ok, _ = run_cmd(
            f"git clone --branch {branch} --single-branch --filter=blob:none --sparse {repo_url} PHDM",
            cwd=str(work_dir)
        )
        if not ok:
            # Fallback : clone complet
            print("   ⚠️ Sparse clone échoué, clone complet...")
            run_cmd(
                f"git clone --branch {branch} --single-branch {repo_url} PHDM",
                cwd=str(work_dir)
            )
        else:
            # Configurer sparse checkout pour les dossiers cibles
            for repo_dir_path in REPO_DIRS:
                run_cmd(f"git sparse-checkout add {repo_dir_path}", cwd=str(repo_dir))
    
    # Configuration git
    run_cmd('git config user.email "colab-sync@phdm.local"', cwd=str(repo_dir))
    run_cmd('git config user.name "PHDM Colab Pipeline"', cwd=str(repo_dir))
    
    return repo_dir


def sync_drive_to_github(repo_dir, drive_allmd, drive_allpdf, branch):
    """Mode PUSH : copie Drive → repo local → commit & push."""
    print("\n📤 MODE PUSH : Drive → GitHub")
    print("=" * 50)
    
    files_copied = 0
    
    # Mapping Drive → Repo
    sync_map = {
        drive_allmd: repo_dir / "ALL" / "ALLMD",
        drive_allpdf: repo_dir / "ALL" / "ALLPDF",
    }
    
    for src, dst in sync_map.items():
        if not src.exists():
            print(f"   ⚠️ Source introuvable: {src}")
            continue
        
        dst.mkdir(parents=True, exist_ok=True)
        print(f"\n   📂 Sync: {src.name} → {dst.relative_to(repo_dir)}")
        
        # Compter les fichiers avant
        src_files = list(src.rglob("*"))
        src_files = [f for f in src_files if f.is_file()]
        print(f"      Fichiers source: {len(src_files)}")
        
        # Utiliser rsync si disponible (plus efficace)
        if shutil.which("rsync"):
            ok, output = run_cmd(
                f'rsync -av --update --progress "{src}/" "{dst}/"',
                check=False
            )
            if ok:
                # Compter les fichiers transférés depuis la sortie rsync
                transferred = [l for l in output.split('\n') if l.strip() and not l.startswith('sending') and not l.startswith('total')]
                files_copied += len(transferred)
            else:
                # Fallback copytree
                shutil.copytree(src, dst, dirs_exist_ok=True)
                files_copied += len(src_files)
        else:
            shutil.copytree(src, dst, dirs_exist_ok=True)
            files_copied += len(src_files)
        
        # Compter les fichiers destination
        dst_files = list(dst.rglob("*"))
        dst_files = [f for f in dst_files if f.is_file()]
        print(f"      Fichiers destination: {len(dst_files)}")
    
    # Git add + commit + push
    print(f"\n   📤 Commit & Push vers {branch}...")
    
    run_cmd("git add ALL/ALLMD/ ALL/ALLPDF/", cwd=str(repo_dir))
    
    # Vérifier s'il y a des changements
    ok, status_output = run_cmd("git status --porcelain", cwd=str(repo_dir), check=False)
    
    if status_output.strip():
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
        commit_msg = f"📤 Sync Drive→GitHub ({timestamp}) — Pipeline ColabV1"
        
        # Compter les changements
        changes = [l for l in status_output.strip().split('\n') if l.strip()]
        added = len([l for l in changes if l.startswith('A') or l.startswith('?')])
        modified = len([l for l in changes if l.startswith('M')])
        deleted = len([l for l in changes if l.startswith('D')])
        
        print(f"      Changements: +{added} ajoutés, ~{modified} modifiés, -{deleted} supprimés")
        
        run_cmd(f'git commit -m "{commit_msg}"', cwd=str(repo_dir))
        
        # Push avec gestion des conflits
        ok, err = run_cmd(f"git push origin {branch}", cwd=str(repo_dir), check=False)
        
        if not ok:
            print(f"   ⚠️ Push échoué, tentative de rebase...")
            run_cmd(f"git pull --rebase origin {branch}", cwd=str(repo_dir))
            ok, err = run_cmd(f"git push origin {branch}", cwd=str(repo_dir), check=False)
            
            if not ok:
                print(f"   ❌ Push échoué après rebase: {err[:200]}")
                return False
        
        print(f"   ✅ Push réussi vers {branch}")
        return True
    else:
        print(f"   ℹ️ Aucun changement détecté — rien à pousser.")
        return True


def sync_github_to_drive(repo_dir, drive_allmd, drive_allpdf):
    """Mode PULL : repo local → copie vers Drive."""
    print("\n📥 MODE PULL : GitHub → Drive")
    print("=" * 50)
    
    # Mapping Repo → Drive
    sync_map = {
        repo_dir / "ALL" / "ALLMD": drive_allmd,
        repo_dir / "ALL" / "ALLPDF": drive_allpdf,
    }
    
    for src, dst in sync_map.items():
        if not src.exists():
            print(f"   ⚠️ Dossier repo introuvable: {src.relative_to(repo_dir)}")
            # Créer le dossier dans le repo si inexistant
            src.mkdir(parents=True, exist_ok=True)
            print(f"      (dossier vide créé)")
            continue
        
        dst.mkdir(parents=True, exist_ok=True)
        
        src_files = [f for f in src.rglob("*") if f.is_file()]
        print(f"\n   📂 Sync: {src.relative_to(repo_dir)} → Drive ({len(src_files)} fichiers)")
        
        if shutil.which("rsync"):
            run_cmd(f'rsync -av --update "{src}/" "{dst}/"', check=False)
        else:
            shutil.copytree(src, dst, dirs_exist_ok=True)
        
        dst_files = [f for f in dst.rglob("*") if f.is_file()]
        print(f"      Fichiers sur Drive: {len(dst_files)}")
    
    print(f"\n   ✅ Synchronisation Drive terminée")
    return True


def get_sync_status(repo_dir, drive_allmd, drive_allpdf):
    """Affiche un résumé de l'état des fichiers des deux côtés."""
    print("\n📊 ÉTAT DE LA SYNCHRONISATION")
    print("=" * 50)
    
    pairs = [
        ("ALLMD", repo_dir / "ALL" / "ALLMD", drive_allmd),
        ("ALLPDF", repo_dir / "ALL" / "ALLPDF", drive_allpdf),
    ]
    
    for name, repo_path, drive_path in pairs:
        repo_count = len([f for f in repo_path.rglob("*") if f.is_file()]) if repo_path.exists() else 0
        drive_count = len([f for f in drive_path.rglob("*") if f.is_file()]) if drive_path.exists() else 0
        
        repo_size = sum(f.stat().st_size for f in repo_path.rglob("*") if f.is_file()) / 1e6 if repo_path.exists() else 0
        drive_size = sum(f.stat().st_size for f in drive_path.rglob("*") if f.is_file()) / 1e6 if drive_path.exists() else 0
        
        status = "✅" if repo_count == drive_count else "⚠️"
        print(f"\n   {status} {name}:")
        print(f"      GitHub : {repo_count:>5} fichiers ({repo_size:>8.1f} MB)")
        print(f"      Drive  : {drive_count:>5} fichiers ({drive_size:>8.1f} MB)")
        
        if repo_count != drive_count:
            diff = abs(repo_count - drive_count)
            direction = "GitHub" if repo_count > drive_count else "Drive"
            print(f"      → {diff} fichiers de plus sur {direction}")


# ═══════════════════════════════════════════════════════════════════════════
# EXÉCUTION
# ═══════════════════════════════════════════════════════════════════════════

print("=" * 60)
print(f"🔄 SYNCHRONISATION DRIVE ↔ GITHUB")
print(f"   Repo   : {GITHUB_REPO}")
print(f"   Branche: {GITHUB_BRANCH}")
print(f"   Mode   : {SYNC_MODE}")
print("=" * 60)

# 1. Clone ou mise à jour du repo
repo_dir = clone_or_update_repo(GIT_WORK_DIR, REPO_URL_AUTH, GITHUB_BRANCH)

# 2. Afficher l'état actuel
get_sync_status(repo_dir, DRIVE_ALLMD, DRIVE_ALLPDF)

# 3. Exécuter la synchronisation selon le mode
if SYNC_MODE.upper() == "PUSH":
    success = sync_drive_to_github(repo_dir, DRIVE_ALLMD, DRIVE_ALLPDF, GITHUB_BRANCH)
elif SYNC_MODE.upper() == "PULL":
    success = sync_github_to_drive(repo_dir, DRIVE_ALLMD, DRIVE_ALLPDF)
else:
    print(f"❌ Mode inconnu: {SYNC_MODE}. Utilisez 'PUSH' ou 'PULL'.")
    success = False

# 4. Afficher l'état après sync
if success:
    get_sync_status(repo_dir, DRIVE_ALLMD, DRIVE_ALLPDF)

# 5. Nettoyage optionnel
CLEANUP_GIT = False  # Mettre True pour supprimer le clone après sync
if CLEANUP_GIT and GIT_WORK_DIR.exists():
    shutil.rmtree(GIT_WORK_DIR)
    print("\n🧹 Dossier git temporaire supprimé.")

print("\n✅ Synchronisation terminée.")